In [ ]:
import os, json, zipfile, math, copy, sys, time, traceback, random
import numpy as np
from pathlib import Path

import onnx
import onnx.helper as oh
import onnx.numpy_helper as onh
from onnx import TensorProto

try:
    import onnxruntime as ort
except ImportError:
    os.system("pip install onnxruntime -q")
    import onnxruntime as ort

# ============================================================
# CONFIG
# ============================================================
TASK_DIR = Path("/kaggle/input/competitions/neurogolf-2026")
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
C, H, W = 10, 30, 30
HW = H * W
CHW = C * H * W
MAX_ARC_GEN_VALIDATE = 50  # Validate on up to this many arc-gen pairs

# ============================================================
# DATA UTILS
# ============================================================
def load_task(path):
    with open(path) as f:
        return json.load(f)

def grid_to_tensor(grid):
    g = np.array(grid, dtype=np.int32)
    h, w = g.shape
    t = np.zeros((1, C, H, W), dtype=np.float32)
    if h > H or w > W:
        return t
    for v in range(C):
        t[0, v, :h, :w] = (g == v).astype(np.float32)
    return t

def tensor_to_grid(t, out_h, out_w):
    arr = t[0]
    out_h = min(out_h, H)
    out_w = min(out_w, W)
    slice_ = arr[:, :out_h, :out_w]
    max_vals = slice_.max(axis=0)
    argmax = slice_.argmax(axis=0)
    grid = np.where(max_vals < 0.5, 0, argmax).astype(int)
    return grid.tolist()

def shapes_match(p):
    return np.array(p["input"]).shape == np.array(p["output"]).shape

# ============================================================
# MODEL CHECK + PROPER COST (WITH MACs)
# ============================================================
def check_model_correct(model, pairs):
    try:
        buf = model.SerializeToString()
        sess = ort.InferenceSession(buf, providers=["CPUExecutionProvider"])
        for p in pairs:
            inp = grid_to_tensor(p["input"])
            out_g = np.array(p["output"])
            oh_, ow_ = out_g.shape
            pred = sess.run(None, {"input": inp})[0]
            pred_grid = tensor_to_grid(pred, oh_, ow_)
            if pred_grid != p["output"]:
                return False
        return True
    except Exception:
        return False

def estimate_model_cost(model):
    """Cost = params + memory_bytes + MACs.
    Critical: V2 was missing MACs! This changes strategy ranking."""
    try:
        total_params = 0
        total_bytes = 0

        # Collect all initializers and constants with their arrays
        tensors = {}  # name → array
        for init in model.graph.initializer:
            arr = onh.to_array(init)
            tensors[init.name] = arr
            total_params += arr.size
            total_bytes += arr.nbytes
        for node in model.graph.node:
            if node.op_type == "Constant":
                for attr in node.attribute:
                    if attr.t and (attr.t.raw_data or list(attr.t.float_data)
                                   or list(attr.t.int64_data) or list(attr.t.int32_data)):
                        arr = onh.to_array(attr.t)
                        if node.output:
                            tensors[node.output[0]] = arr
                        total_params += arr.size
                        total_bytes += arr.nbytes

        # Estimate MACs
        total_macs = 0
        for node in model.graph.node:
            if node.op_type == "Conv":
                if len(node.input) >= 2 and node.input[1] in tensors:
                    w = tensors[node.input[1]]
                    if w.ndim == 4:
                        c_out, c_in, kh, kw = w.shape
                        # Output spatial = input spatial (we use 'same' padding)
                        total_macs += c_out * c_in * kh * kw * H * W
            elif node.op_type == "Gemm":
                if len(node.input) >= 2 and node.input[1] in tensors:
                    w = tensors[node.input[1]]
                    if w.ndim == 2:
                        m_dim = 1  # batch
                        total_macs += m_dim * w.shape[0] * w.shape[1]
            elif node.op_type in ("Mul", "Add", "Max", "Relu"):
                # Element-wise ops — count each elem as 1 "op"
                # Use output size = HW*C for our graphs
                total_macs += CHW
            # Gather, Reshape, Identity: 0 MACs

        return total_params + total_bytes + total_macs
    except Exception as e:
        return float('inf')

# ============================================================
# ONNX BUILDERS (minimized for cost)
# ============================================================

def make_identity_onnx():
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    node = oh.make_node("Identity", ["input"], ["output"])
    graph = oh.make_graph([node], "identity", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


# ============================================================
# CHEAP PRIMITIVES (from the shared notebook)
# These use Transpose/Slice/Pad/Tile/Resize instead of Gather-tables.
# Much cheaper: 0 MACs, tiny bytes, no 900-element index arrays.
# ============================================================

def _const_node(name, arr):
    t = onh.from_array(arr, name=name + "_v")
    return oh.make_node("Constant", [], [name], value=t)

def _slice_reverse_nodes(axes, dims, in_name, out_name, suf=""):
    """Build Slice nodes that reverse along given axes (equivalent to flipping)."""
    starts = [d - 1 for d in dims]
    ends = [-d - 1 for d in dims]
    steps = [-1] * len(dims)
    return [
        _const_node(f"rs_s{suf}", np.asarray(starts, dtype=np.int64)),
        _const_node(f"rs_e{suf}", np.asarray(ends, dtype=np.int64)),
        _const_node(f"rs_a{suf}", np.asarray(axes, dtype=np.int64)),
        _const_node(f"rs_p{suf}", np.asarray(steps, dtype=np.int64)),
        oh.make_node("Slice",
                     [in_name, f"rs_s{suf}", f"rs_e{suf}", f"rs_a{suf}", f"rs_p{suf}"],
                     [out_name]),
    ]

def _pad_to_canvas_nodes(in_name, out_name, out_h, out_w, suf=""):
    pads = np.asarray([0, 0, 0, 0, 0, 0, H - out_h, W - out_w], dtype=np.int64)
    return [
        _const_node(f"pd{suf}", pads),
        oh.make_node("Pad", [in_name, f"pd{suf}"], [out_name], mode="constant"),
    ]

def _crop_nodes(in_name, out_name, h, w, suf=""):
    return [
        _const_node(f"cs{suf}", np.asarray([0, 0], dtype=np.int64)),
        _const_node(f"ce{suf}", np.asarray([h, w], dtype=np.int64)),
        _const_node(f"ca{suf}", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", [in_name, f"cs{suf}", f"ce{suf}", f"ca{suf}"], [out_name]),
    ]

def _build_model_from_nodes(nodes, name):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    graph = oh.make_graph(nodes, name, [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def make_cheap_rot180_full():
    """Full-canvas 180° rotation via Slice reverse. Only valid when input fills
    the full 30×30 canvas (or the bg channel doesn't leak into positions the
    output wouldn't have values at)."""
    return _build_model_from_nodes(
        _slice_reverse_nodes([2, 3], [H, W], "input", "output"),
        "rot180_full")


def make_cheap_rot180_framed(h, w):
    """180° rotation of a h×w content region, output placed at top-left."""
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([2, 3], [h, w], "content", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", h, w, "p")
    return _build_model_from_nodes(nodes, "rot180_framed")


def make_cheap_rot90cw_framed(h, w):
    """CW 90° rotation of a h×w content region. Output is (w, h)."""
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([3], [h], "t", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "rot90cw_framed")


def make_cheap_rot90ccw_framed(h, w):
    """CCW 90° rotation of a h×w content region. Output is (w, h)."""
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2], [w], "t", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "rot90ccw_framed")


def make_cheap_flip_h_framed(h, w):
    """Flip along rows (flipud) of h×w content region."""
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([2], [h], "content", "flipped", "r")
    nodes += _pad_to_canvas_nodes("flipped", "output", h, w, "p")
    return _build_model_from_nodes(nodes, "flip_h_framed")


def make_cheap_flip_w_framed(h, w):
    """Flip along cols (fliplr) of h×w content region."""
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += _slice_reverse_nodes([3], [w], "content", "flipped", "r")
    nodes += _pad_to_canvas_nodes("flipped", "output", h, w, "p")
    return _build_model_from_nodes(nodes, "flip_w_framed")


def make_cheap_transpose_framed(h, w):
    """Transpose of h×w content region. Output is (w, h)."""
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _pad_to_canvas_nodes("t", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "transpose_framed")


def make_cheap_anti_transpose_framed(h, w):
    """Anti-transpose: transpose then flip both axes."""
    nodes = _crop_nodes("input", "content", h, w, "0")
    nodes += [oh.make_node("Transpose", ["content"], ["t"], perm=[0, 1, 3, 2])]
    nodes += _slice_reverse_nodes([2, 3], [w, h], "t", "rotated", "r")
    nodes += _pad_to_canvas_nodes("rotated", "output", w, h, "p")
    return _build_model_from_nodes(nodes, "anti_transpose_framed")


def make_cheap_crop(r0, c0, out_h, out_w):
    """Crop a region at (r0, c0) of size (out_h, out_w), place at top-left."""
    nodes = [
        _const_node("cs", np.asarray([r0, c0], dtype=np.int64)),
        _const_node("ce", np.asarray([r0 + out_h, c0 + out_w], dtype=np.int64)),
        _const_node("ca", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["input", "cs", "ce", "ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas_nodes("cropped", "output", out_h, out_w, "p")
    return _build_model_from_nodes(nodes, "crop_net")


def make_cheap_tile(tr, tc, in_h, in_w):
    """Tile the content region (tr, tc) times along H/W."""
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    repeats = np.asarray([1, 1, tr, tc], dtype=np.int64)
    nodes += [
        _const_node("tile_r", repeats),
        oh.make_node("Tile", ["content", "tile_r"], ["tiled"]),
    ]
    nodes += _pad_to_canvas_nodes("tiled", "output", in_h * tr, in_w * tc, "p")
    return _build_model_from_nodes(nodes, "tile_net")


def make_cheap_upscale(f, in_h, in_w):
    """Nearest-neighbor upscale by factor f."""
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [
        _const_node("rsz_roi", np.asarray([], dtype=np.float32)),
        _const_node("rsz_scales", np.asarray([], dtype=np.float32)),
        _const_node("rsz_sizes", np.asarray([1, C, in_h * f, in_w * f], dtype=np.int64)),
        oh.make_node(
            "Resize",
            ["content", "rsz_roi", "rsz_scales", "rsz_sizes"],
            ["upscaled"],
            mode="nearest",
            coordinate_transformation_mode="asymmetric",
            nearest_mode="floor",
        ),
    ]
    nodes += _pad_to_canvas_nodes("upscaled", "output", in_h * f, in_w * f, "p")
    return _build_model_from_nodes(nodes, "upscale_net")


def make_cheap_concat_flip_w(in_h, in_w):
    """Output = [input | fliplr(input)] of shape (h, 2w)."""
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([3], [in_w], "content", "flipped", "fw")
    nodes += [oh.make_node("Concat", ["content", "flipped"], ["concated"], axis=3)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "concat_flip_w")


def make_cheap_concat_flip_h(in_h, in_w):
    """Output = [input; flipud(input)] of shape (2h, w)."""
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([2], [in_h], "content", "flipped", "fh")
    nodes += [oh.make_node("Concat", ["content", "flipped"], ["concated"], axis=2)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h * 2, in_w, "p")
    return _build_model_from_nodes(nodes, "concat_flip_h")


def make_cheap_quadrant_mirror(in_h, in_w):
    """2×2 quadrant mirror: [content fw; fh rot180] all concatenated."""
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([3], [in_w], "content", "fw", "fw")
    nodes += [oh.make_node("Concat", ["content", "fw"], ["top_row"], axis=3)]
    nodes += _slice_reverse_nodes([2], [in_h], "content", "fh", "fh")
    nodes += _slice_reverse_nodes([2, 3], [in_h, in_w], "content", "fr", "fr")
    nodes += [oh.make_node("Concat", ["fh", "fr"], ["bot_row"], axis=3)]
    nodes += [oh.make_node("Concat", ["top_row", "bot_row"], ["full"], axis=2)]
    nodes += _pad_to_canvas_nodes("full", "output", in_h * 2, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "quadrant_mirror")


def make_cheap_self_concat_h(in_h, in_w):
    """Output = [input; input] of shape (2h, w)."""
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [oh.make_node("Concat", ["content", "content"], ["concated"], axis=2)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h * 2, in_w, "p")
    return _build_model_from_nodes(nodes, "self_concat_h")


def make_cheap_self_concat_w(in_h, in_w):
    """Output = [input | input] of shape (h, 2w)."""
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += [oh.make_node("Concat", ["content", "content"], ["concated"], axis=3)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "self_concat_w")


# ============================================================
# ORIGINAL BUILDERS (kept from v3)
# ============================================================

def make_identity_onnx_v3():
    # duplicate kept for structural symmetry; identity is trivial.
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    node = oh.make_node("Identity", ["input"], ["output"])
    graph = oh.make_graph([node], "identity_v3", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_const_onnx(const_output):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    const_val = const_output.astype(np.float32)
    const_tensor = onh.from_array(const_val, name="const_out")
    const_node = oh.make_node("Constant", [], ["const_out"], value=const_tensor)
    zero_tensor = onh.from_array(np.zeros((1, C, H, W), dtype=np.float32), name="zero_val")
    zero_node = oh.make_node("Constant", [], ["zero_val"], value=zero_tensor)
    mul_node = oh.make_node("Mul", ["input", "zero_val"], ["zeroed"])
    add_node = oh.make_node("Add", ["zeroed", "const_out"], ["output"])
    graph = oh.make_graph([const_node, zero_node, mul_node, add_node], "const_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def _compact_mask(mask):
    """Reduce [1, C, H, W] mask to [1, 1, H, W] when all channels match."""
    if mask.ndim == 4 and mask.shape[0] == 1 and mask.shape[1] == C:
        if np.all(mask[0, 0:1] == mask[0, :]):
            return mask[:, 0:1].copy()
    return mask

def make_gather_onnx(gather_indices, mask=None):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])

    max_idx = int(gather_indices.max()) if gather_indices.size > 0 else 0
    if max_idx < 32768:
        gi_tensor = onh.from_array(gather_indices.astype(np.int32), name="gather_idx")
    else:
        gi_tensor = onh.from_array(gather_indices.astype(np.int64), name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)

    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)

    if mask is not None:
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["gathered_4d"])
        mask = _compact_mask(mask)
        mask_tensor = onh.from_array(mask.astype(np.float32), name="mask")
        mask_node = oh.make_node("Constant", [], ["mask"], value=mask_tensor)
        mul_node = oh.make_node("Mul", ["gathered_4d", "mask"], ["output"])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
                 out_shape_node, reshape_back, mask_node, mul_node]
    else:
        reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["output"])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node, out_shape_node, reshape_back]

    graph = oh.make_graph(nodes, "gather_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_conv1x1_onnx(weight):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    w_tensor = onh.from_array(weight.astype(np.float32), name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    conv_node = oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                             kernel_shape=[1, 1], pads=[0, 0, 0, 0])
    graph = oh.make_graph([w_node, conv_node], "conv1x1", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_channel_gather_onnx(gather_ch):
    """Color map via channel gather. Only works for permutations (bijective maps).
    Cost: ~50 vs 90500 for 1x1 conv. Score: ~20.5 vs 13.6. MASSIVE win.
    """
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    gi_tensor = onh.from_array(gather_ch.astype(np.int32), name="gi")
    gi_node = oh.make_node("Constant", [], ["gi"], value=gi_tensor)
    gather = oh.make_node("Gather", ["input", "gi"], ["output"], axis=1)
    graph = oh.make_graph([gi_node, gather], "ch_gather", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_spatial_then_channel_gather_onnx(spatial_idx, channel_idx):
    """Two gathers: spatial then channel. Used for composite transforms."""
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])

    si = onh.from_array(spatial_idx.astype(np.int32), name="si")
    si_node = oh.make_node("Constant", [], ["si"], value=si)
    spatial_gather = oh.make_node("Gather", ["inp_chw", "si"], ["spatial_out"], axis=2)

    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_back = oh.make_node("Reshape", ["spatial_out", "out_shape"], ["spatial_4d"])

    ci = onh.from_array(channel_idx.astype(np.int32), name="ci")
    ci_node = oh.make_node("Constant", [], ["ci"], value=ci)
    channel_gather = oh.make_node("Gather", ["spatial_4d", "ci"], ["output"], axis=1)

    graph = oh.make_graph([shape_chw_node, inp_reshape, si_node, spatial_gather,
                           out_shape_node, reshape_back, ci_node, channel_gather],
                          "spatial_ch_gather", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_conv_onnx(weight, bias=None, kernel_size=3):
    pad = kernel_size // 2
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    w_tensor = onh.from_array(weight.astype(np.float32), name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)
    if bias is not None:
        b_tensor = onh.from_array(bias.astype(np.float32), name="conv_bias")
        b_node = oh.make_node("Constant", [], ["conv_bias"], value=b_tensor)
        conv_node = oh.make_node("Conv", ["input", "conv_weight", "conv_bias"], ["output"],
                                 kernel_shape=[kernel_size, kernel_size],
                                 pads=[pad, pad, pad, pad])
        nodes = [w_node, b_node, conv_node]
    else:
        conv_node = oh.make_node("Conv", ["input", "conv_weight"], ["output"],
                                 kernel_shape=[kernel_size, kernel_size],
                                 pads=[pad, pad, pad, pad])
        nodes = [w_node, conv_node]
    graph = oh.make_graph(nodes, "conv_net", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_gather_then_conv1x1_onnx(gather_indices, conv_weight, mask=None):
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input", "shape_chw"], ["inp_chw"])
    gi_tensor = onh.from_array(gather_indices.astype(np.int32), name="gather_idx")
    gi_node = oh.make_node("Constant", [], ["gather_idx"], value=gi_tensor)
    gather_node = oh.make_node("Gather", ["inp_chw", "gather_idx"], ["gathered"], axis=2)
    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)
    reshape_back = oh.make_node("Reshape", ["gathered", "out_shape"], ["gathered_4d"])

    w_tensor = onh.from_array(conv_weight.astype(np.float32), name="conv_weight")
    w_node = oh.make_node("Constant", [], ["conv_weight"], value=w_tensor)

    if mask is not None:
        mask = _compact_mask(mask)
        mask_tensor = onh.from_array(mask.astype(np.float32), name="mask")
        mask_node = oh.make_node("Constant", [], ["mask"], value=mask_tensor)
        mask_mul = oh.make_node("Mul", ["gathered_4d", "mask"], ["masked"])
        conv_node = oh.make_node("Conv", ["masked", "conv_weight"], ["output"],
                                 kernel_shape=[1, 1], pads=[0, 0, 0, 0])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
                 out_shape_node, reshape_back, mask_node, mask_mul, w_node, conv_node]
    else:
        conv_node = oh.make_node("Conv", ["gathered_4d", "conv_weight"], ["output"],
                                 kernel_shape=[1, 1], pads=[0, 0, 0, 0])
        nodes = [shape_chw_node, inp_reshape, gi_node, gather_node,
                 out_shape_node, reshape_back, w_node, conv_node]

    graph = oh.make_graph(nodes, "gather_conv1x1", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

def make_symmetry_completion_onnx(gather_index_list, mask=None):
    """MAX over multiple Gather ops, with channel-0 (bg) suppressed."""
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])

    ch_mask = np.ones((1, C, 1, 1), dtype=np.float32); ch_mask[0, 0, 0, 0] = 0.0
    ch_mask_tensor = onh.from_array(ch_mask, name="ch_mask")
    ch_mask_node = oh.make_node("Constant", [], ["ch_mask"], value=ch_mask_tensor)
    nobg_mul = oh.make_node("Mul", ["input", "ch_mask"], ["input_nobg"])

    shape_chw = onh.from_array(np.array([1, C, HW], dtype=np.int64), name="shape_chw")
    shape_chw_node = oh.make_node("Constant", [], ["shape_chw"], value=shape_chw)
    inp_reshape = oh.make_node("Reshape", ["input_nobg", "shape_chw"], ["inp_chw"])
    out_shape = onh.from_array(np.array([1, C, H, W], dtype=np.int64), name="out_shape")
    out_shape_node = oh.make_node("Constant", [], ["out_shape"], value=out_shape)

    nodes = [ch_mask_node, nobg_mul, shape_chw_node, inp_reshape, out_shape_node]
    reshape_names = []
    for i, gi in enumerate(gather_index_list):
        gi_name = f"gather_idx_{i}"
        gi_tensor = onh.from_array(gi.astype(np.int32), name=gi_name)
        gi_node = oh.make_node("Constant", [], [gi_name], value=gi_tensor)
        gname = f"gathered_{i}"
        gather_node = oh.make_node("Gather", ["inp_chw", gi_name], [gname], axis=2)
        rname = f"reshaped_{i}"
        reshape_node = oh.make_node("Reshape", [gname, "out_shape"], [rname])
        nodes.extend([gi_node, gather_node, reshape_node])
        reshape_names.append(rname)

    if len(reshape_names) == 1:
        final = reshape_names[0]
    else:
        prev = reshape_names[0]
        for i in range(1, len(reshape_names)):
            out_name = "output" if (i == len(reshape_names) - 1 and mask is None) else f"max_{i}"
            max_node = oh.make_node("Max", [prev, reshape_names[i]], [out_name])
            nodes.append(max_node)
            prev = out_name
        final = prev

    if mask is not None:
        mask = _compact_mask(mask)
        mask_tensor = onh.from_array(mask.astype(np.float32), name="mask")
        mask_node = oh.make_node("Constant", [], ["mask"], value=mask_tensor)
        mul_node = oh.make_node("Mul", [final, "mask"], ["output"])
        nodes.extend([mask_node, mul_node])
    elif len(reshape_names) == 1:
        nodes.append(oh.make_node("Identity", [final], ["output"]))

    graph = oh.make_graph(nodes, "sym_complete", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model

# ============================================================
# ANALYSIS
# ============================================================
def analyze_transformation(pairs):
    info = {}
    in_sizes = [np.array(p["input"]).shape for p in pairs]
    out_sizes = [np.array(p["output"]).shape for p in pairs]
    info["in_sizes"] = in_sizes
    info["out_sizes"] = out_sizes
    info["same_size"] = all(i == o for i, o in zip(in_sizes, out_sizes))
    info["const_output"] = len(set(map(lambda p: json.dumps(p["output"]), pairs))) == 1
    info["all_same_in_size"] = len(set(in_sizes)) == 1
    info["all_same_out_size"] = len(set(out_sizes)) == 1

    if info["same_size"]:
        diffs = [np.sum(np.array(p["input"]) != np.array(p["output"])) for p in pairs]
        info["identity"] = all(d == 0 for d in diffs)
    else:
        info["identity"] = False

    color_maps = []
    for p in pairs:
        ig = np.array(p["input"]).flatten()
        og = np.array(p["output"]).flatten()
        if ig.shape == og.shape:
            cmap = {}; valid = True
            for a, b in zip(ig, og):
                if a in cmap and cmap[a] != b:
                    valid = False; break
                cmap[a] = b
            if valid:
                color_maps.append(cmap)
    if color_maps and len(color_maps) == len(pairs):
        common = color_maps[0]
        for cm in color_maps[1:]:
            if cm != common: common = None; break
        info["color_map"] = common
    else:
        info["color_map"] = None
    return info

# ============================================================
# DETECTORS
# ============================================================

def detect_rotation(pairs):
    for angle in [90, 180, 270]:
        k = angle // 90
        valid = True
        for p in pairs:
            ig = np.array(p["input"]); og = np.array(p["output"])
            rotated = np.rot90(ig, k)
            if rotated.shape != og.shape or not np.array_equal(rotated, og):
                valid = False; break
        if valid: return angle
    return None

def detect_flip(pairs):
    for axis_val, name in [(0, "vertical"), (1, "horizontal"), (-1, "both")]:
        valid = True
        for p in pairs:
            ig = np.array(p["input"]); og = np.array(p["output"])
            if axis_val == -1:
                flipped = np.flip(np.flip(ig, 0), 1)
            else:
                flipped = np.flip(ig, axis=axis_val)
            if flipped.shape != og.shape or not np.array_equal(flipped, og):
                valid = False; break
        if valid: return name
    return None

def detect_transpose(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.T.shape != og.shape or not np.array_equal(ig.T, og):
            return False
    return True

def detect_anti_transpose(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        anti = np.flip(np.flip(ig.T, 0), 1)
        if anti.shape != og.shape or not np.array_equal(anti, og):
            return False
    return True

def detect_crop(pairs):
    results = []
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ > ih or ow_ > iw: return None
        found = False
        for r in range(ih - oh_ + 1):
            for c in range(iw - ow_ + 1):
                if np.array_equal(ig[r:r+oh_, c:c+ow_], og):
                    results.append((r, c, oh_, ow_))
                    found = True; break
            if found: break
        if not found: return None
    if len(results) == len(pairs):
        offsets = [(r[0], r[1]) for r in results]
        if len(set(offsets)) == 1:
            return (offsets[0][0], offsets[0][1], results[0][2], results[0][3])
    return None

def detect_tile(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ % ih != 0 or ow_ % iw != 0: return None
        tr = oh_ // ih; tc = ow_ // iw
        if not np.array_equal(np.tile(ig, (tr, tc)), og): return None
    ig0 = np.array(pairs[0]["input"]); og0 = np.array(pairs[0]["output"])
    return (og0.shape[0] // ig0.shape[0], og0.shape[1] // ig0.shape[1])

def detect_scale(pairs):
    for factor in [2, 3, 4, 5]:
        valid = True
        for p in pairs:
            ig = np.array(p["input"]); og = np.array(p["output"])
            ih, iw = ig.shape; oh_, ow_ = og.shape
            if oh_ != ih * factor or ow_ != iw * factor:
                valid = False; break
            if not np.array_equal(np.repeat(np.repeat(ig, factor, 0), factor, 1), og):
                valid = False; break
        if valid: return factor
    return None

def detect_upscale(pairs):
    for yf, xf in [(1,2),(2,1),(1,3),(3,1),(2,3),(3,2),(1,4),(4,1)]:
        valid = True
        for p in pairs:
            ig = np.array(p["input"]); og = np.array(p["output"])
            ih, iw = ig.shape; oh_, ow_ = og.shape
            if oh_ != ih * yf or ow_ != iw * xf:
                valid = False; break
            if not np.array_equal(np.repeat(np.repeat(ig, yf, 0), xf, 1), og):
                valid = False; break
        if valid: return (yf, xf)
    return None

def detect_color_replace(pairs):
    if not all(shapes_match(p) for p in pairs): return None
    replacements = {}
    for p in pairs:
        ig = np.array(p["input"]).flatten()
        og = np.array(p["output"]).flatten()
        for a, b in zip(ig, og):
            if a != b:
                if a in replacements and replacements[a] != b:
                    return None
                replacements[a] = b
    if not replacements: return None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        expected = np.copy(ig)
        for src, dst in replacements.items():
            expected[ig == src] = dst
        if not np.array_equal(expected, og): return None
    return replacements

def detect_color_swap(pairs):
    """Only accept swap if we have evidence of BOTH directions. E.g., we need
    at least one pair where 'a' appears in input (becomes 'b' in output),
    AND at least one pair where 'b' appears in input (becomes 'a' in output).
    Otherwise it's ambiguous with a simple replace."""
    if not all(shapes_match(p) for p in pairs): return None
    swap_pairs = set()
    for p in pairs:
        ig = np.array(p["input"]).flatten(); og = np.array(p["output"]).flatten()
        for a, b in zip(ig, og):
            if a != b:
                swap_pairs.add((min(a,b), max(a,b)))
    if len(swap_pairs) != 1: return None
    a, b = list(swap_pairs)[0]

    saw_a_to_b = False
    saw_b_to_a = False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        expected = np.copy(ig)
        expected[ig == a] = b; expected[ig == b] = a
        if not np.array_equal(expected, og): return None
        # Evidence requires a pixel with input==a mapping to output==b AND vice versa
        if np.any((ig == a) & (og == b)): saw_a_to_b = True
        if np.any((ig == b) & (og == a)): saw_b_to_a = True
    if not (saw_a_to_b and saw_b_to_a):
        return None  # Ambiguous — could be a replace, not a swap
    return (a, b)

def detect_pixel_permutation(pairs):
    """CONSERVATIVE: requires consistent mapping across ALL pairs."""
    if not all(shapes_match(p) for p in pairs): return None
    if len(set(np.array(p["input"]).shape for p in pairs)) != 1: return None
    ih, iw = np.array(pairs[0]["input"]).shape
    if ih > H or iw > W: return None

    # For each destination position, find which source matches across ALL pairs
    n_pairs = len(pairs)
    all_inputs = [np.array(p["input"]).flatten() for p in pairs]
    all_outputs = [np.array(p["output"]).flatten() for p in pairs]

    grid_size = ih * iw
    assignment = {}
    for dst_idx in range(grid_size):
        # What are the target values at dst_idx across pairs?
        targets = [all_outputs[pi][dst_idx] for pi in range(n_pairs)]
        # Find src_idx such that all_inputs[pi][src_idx] == targets[pi] for all pi
        candidates = None
        for pi in range(n_pairs):
            match = set(np.where(all_inputs[pi] == targets[pi])[0].tolist())
            candidates = match if candidates is None else candidates & match
            if not candidates: return None
        if len(candidates) != 1: return None
        assignment[dst_idx] = candidates.pop()

    gather_indices = np.arange(HW, dtype=np.int64)
    for dst_idx, src_idx in assignment.items():
        dst_r, dst_c = divmod(dst_idx, iw)
        src_r, src_c = divmod(src_idx, iw)
        gather_indices[dst_r * W + dst_c] = src_r * W + src_c

    # Final verification
    for p in pairs:
        ig = np.array(p["input"])
        padded = np.zeros((H, W), dtype=np.int32)
        padded[:ih, :iw] = ig
        flat = padded.flatten()
        pred = flat[gather_indices].reshape(H, W)[:ih, :iw]
        if not np.array_equal(pred, np.array(p["output"])):
            return None
    return gather_indices

def detect_mirror_h_concat(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ != ih or ow_ != 2*iw: return False
        if not np.array_equal(np.concatenate([ig, np.flip(ig, 1)], 1), og): return False
    return True

def detect_mirror_v_concat(pairs):
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ != 2*ih or ow_ != iw: return False
        if not np.array_equal(np.concatenate([ig, np.flip(ig, 0)], 0), og): return False
    return True

def detect_self_concat_h(pairs):
    """Output = [input, input]."""
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ != ih or ow_ != 2*iw: return False
        if not np.array_equal(np.concatenate([ig, ig], 1), og): return False
    return True

def detect_self_concat_v(pairs):
    """Output = [input; input]."""
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ != 2*ih or ow_ != iw: return False
        if not np.array_equal(np.concatenate([ig, ig], 0), og): return False
    return True

def detect_quad_mirror(pairs):
    patterns = []
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ != 2*ih or ow_ != 2*iw: return None
        exp1 = np.block([[ig, np.flip(ig,1)], [np.flip(ig,0), np.flip(np.flip(ig,0),1)]])
        if np.array_equal(exp1, og): patterns.append("tl"); continue
        exp2 = np.block([[np.flip(np.flip(ig,0),1), np.flip(ig,0)], [np.flip(ig,1), ig]])
        if np.array_equal(exp2, og): patterns.append("br"); continue
        return None
    if patterns and len(set(patterns)) == 1:
        return patterns[0]
    return None

def detect_nonzero_recolor(pairs):
    """Detect: all non-bg pixels get same color; bg stays bg."""
    if not all(shapes_match(p) for p in pairs): return None
    target_color = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        # bg must stay bg
        if not np.all(og[ig == 0] == 0): return None
        # all non-bg pixels → single color
        nonbg = og[ig != 0]
        if nonbg.size == 0: continue
        unique = np.unique(nonbg)
        if len(unique) != 1: return None
        if target_color is None:
            target_color = unique[0]
        elif target_color != unique[0]:
            return None
    return target_color

def detect_bg_recolor(pairs):
    """Detect: bg pixels (0) → other color, non-bg stay same."""
    if not all(shapes_match(p) for p in pairs): return None
    target = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        # non-bg must stay same
        if not np.array_equal(og[ig != 0], ig[ig != 0]): return None
        # bg → target
        bg_out = og[ig == 0]
        if bg_out.size == 0: continue
        u = np.unique(bg_out)
        if len(u) != 1: return None
        if target is None:
            target = u[0]
        elif target != u[0]:
            return None
    return target

def detect_h_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        expected = np.where(ig != 0, ig, np.flip(ig, 1))
        if not np.array_equal(expected, og): return False
    return True

def detect_v_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        expected = np.where(ig != 0, ig, np.flip(ig, 0))
        if not np.array_equal(expected, og): return False
    return True

def detect_both_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        m_h = np.flip(ig, 1); m_v = np.flip(ig, 0); m_b = np.flip(np.flip(ig,0),1)
        stack = np.stack([ig, m_h, m_v, m_b], 0)
        expected = np.where(ig != 0, ig, stack.max(0))
        if not np.array_equal(expected, og): return False
    return True

def detect_diag_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0] != ig.shape[1]: return False
        expected = np.where(ig != 0, ig, ig.T)
        if not np.array_equal(expected, og): return False
    return True

def detect_all4_symmetry_complete(pairs):
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0] != ig.shape[1]: return False
        m_h = np.flip(ig, 1); m_v = np.flip(ig, 0); m_b = np.flip(np.flip(ig,0),1); m_t = ig.T
        stack = np.stack([ig, m_h, m_v, m_b, m_t], 0)
        expected = np.where(ig != 0, ig, stack.max(0))
        if not np.array_equal(expected, og): return False
    return True

def detect_grid_of_grids_fixed(pairs):
    """Output is always the same block position."""
    block_selections = []
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape; oh_, ow_ = og.shape
        if oh_ >= ih or ow_ >= iw: return None
        if ih % oh_ != 0 or iw % ow_ != 0: return None
        blocks_r = ih // oh_; blocks_c = iw // ow_
        found = None
        for br in range(blocks_r):
            for bc in range(blocks_c):
                if np.array_equal(ig[br*oh_:(br+1)*oh_, bc*ow_:(bc+1)*ow_], og):
                    found = (br, bc); break
            if found: break
        if found is None: return None
        block_selections.append(found + (oh_, ow_))
    if len(set(block_selections)) == 1:
        br, bc, oh_, ow_ = block_selections[0]
        return (br, bc, oh_, ow_)
    return None

# ============================================================
# V8 NEW DETECTORS (from the 351 LB notebook)
# ============================================================

def detect_translation(pairs):
    """Detect if output = input shifted by (dr, dc), zeros fill vacated cells."""
    if not all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs):
        return None
    first_in = np.array(pairs[0]["input"])
    ih, iw = first_in.shape
    rng_r = min(ih, 11)
    rng_c = min(iw, 11)
    for dr in range(-rng_r + 1, rng_r):
        for dc in range(-rng_c + 1, rng_c):
            if dr == 0 and dc == 0:
                continue
            ok = True
            for p in pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                hh, ww = ig.shape
                shifted = np.zeros_like(ig)
                src_r0 = max(0, -dr); src_r1 = min(hh, hh - dr)
                src_c0 = max(0, -dc); src_c1 = min(ww, ww - dc)
                dst_r0 = max(0, dr);  dst_c0 = max(0, dc)
                rh = src_r1 - src_r0; rw = src_c1 - src_c0
                if rh > 0 and rw > 0:
                    shifted[dst_r0:dst_r0+rh, dst_c0:dst_c0+rw] = \
                        ig[src_r0:src_r1, src_c0:src_c1]
                if not np.array_equal(shifted, og):
                    ok = False; break
            if ok:
                return (dr, dc)
    return None


def make_cheap_framed_translation(grid_h, grid_w, dr, dc):
    """Translate content within [0:grid_h, 0:grid_w] by (dr, dc). Slice+Pad, 0 MACs."""
    src_r0 = max(0, -dr); src_r1 = min(grid_h, grid_h - dr)
    src_c0 = max(0, -dc); src_c1 = min(grid_w, grid_w - dc)
    dst_r0 = max(0, dr);  dst_c0 = max(0, dc)
    rh = src_r1 - src_r0; rw = src_c1 - src_c0
    if rh <= 0 or rw <= 0:
        return None
    nodes = [
        _const_node("fts", np.asarray([src_r0, src_c0], dtype=np.int64)),
        _const_node("fte", np.asarray([src_r1, src_c1], dtype=np.int64)),
        _const_node("fta", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["input", "fts", "fte", "fta"], ["fsliced"]),
    ]
    pads = np.asarray([
        0, 0, dst_r0, dst_c0,
        0, 0, H - dst_r0 - rh, W - dst_c0 - rw
    ], dtype=np.int64)
    nodes += [
        _const_node("fpp", pads),
        oh.make_node("Pad", ["fsliced", "fpp"], ["output"], mode="constant"),
    ]
    return _build_model_from_nodes(nodes, "framed_translate")


def detect_downscale(pairs):
    """Downsample by taking every f-th pixel."""
    if not pairs: return None
    for f in [2, 3, 4, 5]:
        ok = True
        for p in pairs:
            ig = np.array(p["input"]); og = np.array(p["output"])
            ih, iw = ig.shape; oh_, ow_ = og.shape
            if ih != oh_ * f or iw != ow_ * f:
                ok = False; break
            if not np.array_equal(ig[::f, ::f], og):
                ok = False; break
        if ok: return f
    return None


def make_cheap_downscale(f, in_h, in_w):
    """Take every f-th pixel. Slice with step, 0 MACs."""
    out_h = in_h // f; out_w = in_w // f
    nodes = [
        _const_node("ds_s", np.asarray([0, 0], dtype=np.int64)),
        _const_node("ds_e", np.asarray([in_h, in_w], dtype=np.int64)),
        _const_node("ds_a", np.asarray([2, 3], dtype=np.int64)),
        _const_node("ds_p", np.asarray([f, f], dtype=np.int64)),
        oh.make_node("Slice", ["input", "ds_s", "ds_e", "ds_a", "ds_p"], ["downsampled"]),
    ]
    nodes += _pad_to_canvas_nodes("downsampled", "output", out_h, out_w, "p")
    return _build_model_from_nodes(nodes, "downscale")


def detect_color_filter(pairs):
    """Colors that get replaced with 0, everything else unchanged."""
    if not all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs):
        return None
    candidate = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        diff = (ig != og)
        if not np.any(diff): continue
        if not np.all(og[diff] == 0): return None
        local = set(int(x) for x in ig[diff].tolist())
        candidate = local if candidate is None else candidate | local
    if not candidate: return None
    for p in pairs:
        ig = np.array(p["input"]).copy()
        for c_ in candidate: ig[ig == c_] = 0
        if not np.array_equal(ig, np.array(p["output"])): return None
    return candidate


def detect_row_broadcast(pairs):
    """Every row of output == some specific row of input (same index for all pairs)."""
    if not all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs):
        return None
    chosen_row = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape
        matched = None
        for r in range(ih):
            if np.all(og == ig[r, :]): matched = r; break
        if matched is None: return None
        if chosen_row is None: chosen_row = matched
        elif chosen_row != matched: return None
    return chosen_row


def detect_col_broadcast(pairs):
    if not all(np.array(p["input"]).shape == np.array(p["output"]).shape for p in pairs):
        return None
    chosen_col = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape
        matched = None
        for c in range(iw):
            if np.all(og == ig[:, c:c+1]): matched = c; break
        if matched is None: return None
        if chosen_col is None: chosen_col = matched
        elif chosen_col != matched: return None
    return chosen_col


def make_cheap_row_broadcast(in_h, in_w, row_idx=0):
    starts = np.asarray([row_idx], dtype=np.int64)
    ends = np.asarray([row_idx + 1], dtype=np.int64)
    axes = np.asarray([2], dtype=np.int64)
    nodes = [
        _const_node("rb_s", starts), _const_node("rb_e", ends), _const_node("rb_a", axes),
        oh.make_node("Slice", ["input", "rb_s", "rb_e", "rb_a"], ["one_row"]),
        _const_node("rb_r", np.asarray([1, 1, in_h, 1], dtype=np.int64)),
        oh.make_node("Tile", ["one_row", "rb_r"], ["tiled"]),
        _const_node("rb_cs", np.asarray([0, 0], dtype=np.int64)),
        _const_node("rb_ce", np.asarray([in_h, in_w], dtype=np.int64)),
        _const_node("rb_ca", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["tiled", "rb_cs", "rb_ce", "rb_ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas_nodes("cropped", "output", in_h, in_w, "p")
    return _build_model_from_nodes(nodes, "row_broadcast")


def make_cheap_col_broadcast(in_h, in_w, col_idx=0):
    starts = np.asarray([col_idx], dtype=np.int64)
    ends = np.asarray([col_idx + 1], dtype=np.int64)
    axes = np.asarray([3], dtype=np.int64)
    nodes = [
        _const_node("cb_s", starts), _const_node("cb_e", ends), _const_node("cb_a", axes),
        oh.make_node("Slice", ["input", "cb_s", "cb_e", "cb_a"], ["one_col"]),
        _const_node("cb_r", np.asarray([1, 1, 1, in_w], dtype=np.int64)),
        oh.make_node("Tile", ["one_col", "cb_r"], ["tiled"]),
        _const_node("cb_cs", np.asarray([0, 0], dtype=np.int64)),
        _const_node("cb_ce", np.asarray([in_h, in_w], dtype=np.int64)),
        _const_node("cb_ca", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["tiled", "cb_cs", "cb_ce", "cb_ca"], ["cropped"]),
    ]
    nodes += _pad_to_canvas_nodes("cropped", "output", in_h, in_w, "p")
    return _build_model_from_nodes(nodes, "col_broadcast")


def detect_border_add(pairs):
    """Output = input with a uniform border of thickness b, color fill."""
    if not pairs: return None
    first_in = np.array(pairs[0]["input"]); first_out = np.array(pairs[0]["output"])
    ih, iw = first_in.shape; oh_, ow_ = first_out.shape
    if (oh_ - ih) != (ow_ - iw): return None
    delta = oh_ - ih
    if delta <= 0 or delta % 2 != 0: return None
    b = delta // 2
    if not np.array_equal(first_out[b:b+ih, b:b+iw], first_in): return None
    vals = set(first_out[:b, :].flatten().tolist()) | set(first_out[-b:, :].flatten().tolist())
    vals |= set(first_out[:, :b].flatten().tolist()) | set(first_out[:, -b:].flatten().tolist())
    if len(vals) != 1: return None
    fill = int(vals.pop())
    for p in pairs[1:]:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih2, iw2 = ig.shape
        if og.shape != (ih2 + 2*b, iw2 + 2*b): return None
        if not np.array_equal(og[b:b+ih2, b:b+iw2], ig): return None
        m = np.zeros(og.shape, dtype=bool)
        m[:b, :] = True; m[-b:, :] = True; m[:, :b] = True; m[:, -b:] = True
        if not np.all(og[m] == fill): return None
    return (b, fill)


def make_cheap_border_add(grid_h, grid_w, border, fill_color):
    """Add border using Pad + constant mask Add. 0 MACs (just Add)."""
    oh_ = grid_h + 2 * border; ow_ = grid_w + 2 * border
    if oh_ > H or ow_ > W: return None
    nodes = _crop_nodes("input", "content", grid_h, grid_w, "pb0")
    pads_border = np.asarray([0, 0, border, border, 0, 0, border, border], dtype=np.int64)
    nodes += [_const_node("pb_pads", pads_border),
              oh.make_node("Pad", ["content", "pb_pads"], ["padded"], mode="constant")]
    mask = np.zeros((1, C, oh_, ow_), dtype=np.float32)
    if 0 <= fill_color < C:
        mask[0, fill_color, :border, :] = 1.0; mask[0, fill_color, -border:, :] = 1.0
        mask[0, fill_color, :, :border] = 1.0; mask[0, fill_color, :, -border:] = 1.0
    nodes += [_const_node("pb_mask", mask), oh.make_node("Add", ["padded", "pb_mask"], ["bordered"])]
    nodes += _pad_to_canvas_nodes("bordered", "output", oh_, ow_, "pbf")
    return _build_model_from_nodes(nodes, "border_add")


def detect_crop_plus_color(pairs):
    """Output = crop + color remap."""
    if not pairs: return None
    first_in = np.array(pairs[0]["input"]); first_out = np.array(pairs[0]["output"])
    oh_, ow_ = first_out.shape; ih, iw = first_in.shape
    if oh_ > ih or ow_ > iw: return None
    for r0 in range(ih - oh_ + 1):
        for c0 in range(iw - ow_ + 1):
            cmap = {}; valid = True
            for p in pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                if og.shape != (oh_, ow_): valid = False; break
                if r0 + oh_ > ig.shape[0] or c0 + ow_ > ig.shape[1]: valid = False; break
                region = ig[r0:r0+oh_, c0:c0+ow_]
                for a, b in zip(region.flatten(), og.flatten()):
                    a, b = int(a), int(b)
                    if a in cmap and cmap[a] != b: valid = False; break
                    cmap[a] = b
                if not valid: break
            if not valid: continue
            if all(k == v for k, v in cmap.items()): continue
            ok = True
            for p in pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                region = ig[r0:r0+oh_, c0:c0+ow_].copy()
                for k, v in cmap.items(): region[region == k] = v
                if not np.array_equal(region, og): ok = False; break
            if ok: return (r0, c0, oh_, ow_, cmap)
    return None


def detect_concat_rot180_w(pairs):
    """Output = [input | rot180(input)]."""
    if not pairs: return None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        ih, iw = ig.shape
        if og.shape != (ih, iw * 2): return None
        if not np.array_equal(np.concatenate([ig, np.rot90(ig, 2)], axis=1), og): return None
    return True


def make_cheap_concat_rot180_w(in_h, in_w):
    nodes = _crop_nodes("input", "content", in_h, in_w, "0")
    nodes += _slice_reverse_nodes([2, 3], [in_h, in_w], "content", "rotated", "r")
    nodes += [oh.make_node("Concat", ["content", "rotated"], ["concated"], axis=3)]
    nodes += _pad_to_canvas_nodes("concated", "output", in_h, in_w * 2, "p")
    return _build_model_from_nodes(nodes, "concat_rot180_w")


def make_cheap_crop_then_color(r0, c0, out_h, out_w, color_map):
    """Crop then channel-axis Gather for color remap."""
    nodes = [
        _const_node("ctc_s", np.asarray([r0, c0], dtype=np.int64)),
        _const_node("ctc_e", np.asarray([r0 + out_h, c0 + out_w], dtype=np.int64)),
        _const_node("ctc_a", np.asarray([2, 3], dtype=np.int64)),
        oh.make_node("Slice", ["input", "ctc_s", "ctc_e", "ctc_a"], ["cropped"]),
    ]
    inv = {d: s for s, d in color_map.items()}
    idx = np.arange(C, dtype=np.int64)
    mask = np.ones(C, dtype=np.float32)
    src_away = {s for s, d in color_map.items() if s != d}
    for d in range(C):
        if d in inv: idx[d] = inv[d]
        elif d in src_away: idx[d] = 0; mask[d] = 0.0
    nodes += [_const_node("gi_ctc", idx)]
    if mask.min() == 1.0:
        nodes += [oh.make_node("Gather", ["cropped", "gi_ctc"], ["colored"], axis=1)]
    else:
        nodes += [oh.make_node("Gather", ["cropped", "gi_ctc"], ["g_ctc"], axis=1),
                  _const_node("cm_ctc", mask.reshape(1, C, 1, 1)),
                  oh.make_node("Mul", ["g_ctc", "cm_ctc"], ["colored"])]
    nodes += _pad_to_canvas_nodes("colored", "output", out_h, out_w, "pctc")
    return _build_model_from_nodes(nodes, "crop_color")


# ============================================================
# V8 SPATIAL ZERO-AMBIGUITY (from the 351 LB notebook)
# ============================================================

def try_spatial_zero_ambiguity(all_pairs):
    """Find pixel-level spatial mapping across ALL pairs simultaneously.
    For each output position, find the unique input position whose channel
    signature matches across every pair. Only accepts zero-ambiguity solutions.
    Returns a masked gather model or None."""
    for p in all_pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        if ig.shape[0] > H or ig.shape[1] > W or og.shape[0] > H or og.shape[1] > W:
            return None

    inputs = []
    outputs = []
    for p in all_pairs:
        inputs.append(grid_to_tensor(p["input"])[0].reshape(C, -1))
        outputs.append(grid_to_tensor(p["output"])[0].reshape(C, -1))
    inputs = np.stack(inputs, axis=0)   # (N, C, HW)
    outputs = np.stack(outputs, axis=0)  # (N, C, HW)

    # Build lookup: channel signature -> list of input positions
    input_lookup = {}
    for src_idx in range(HW):
        sig = inputs[:, :, src_idx].tobytes()
        input_lookup.setdefault(sig, []).append(src_idx)

    gather_indices = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    has_content = False

    for out_idx in range(HW):
        sig_array = outputs[:, :, out_idx]
        if np.all(sig_array == 0.0):
            continue  # padding position, leave as SAFE_PAD_IDX
        sig = sig_array.tobytes()
        candidates = input_lookup.get(sig)
        if not candidates:
            return None  # no input position matches this output
        if len(candidates) > 1:
            return None  # ambiguous — multiple input positions could map here
        gather_indices[out_idx] = candidates[0]
        has_content = True

    if not has_content:
        return None

    # Build gather model (no mask needed thanks to SAFE_PAD_IDX)
    model = make_gather_onnx(gather_indices)
    return model

SAFE_PAD_IDX = HW - 1  # Gather here for "don't care" regions; input[H-1,W-1] is padded (all-zero in one-hot)

def build_rotation_gather(angle, ih, iw):
    """Build gather for np.rot90(input, k). Unused positions point to safe padding."""
    k = angle // 90
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    if k == 1:
        for r in range(iw):
            for c in range(ih):
                idx[r*W + c] = c*W + (iw - 1 - r)
    elif k == 2:
        for r in range(ih):
            for c in range(iw):
                idx[r*W + c] = (ih-1-r)*W + (iw-1-c)
    elif k == 3:
        for r in range(iw):
            for c in range(ih):
                idx[r*W + c] = (ih-1-c)*W + r
    return idx

def build_flip_gather(direction, ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            if direction == "vertical":
                idx[r*W+c] = (ih-1-r)*W + c
            elif direction == "horizontal":
                idx[r*W+c] = r*W + (iw-1-c)
            else:
                idx[r*W+c] = (ih-1-r)*W + (iw-1-c)
    return idx

def build_transpose_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(iw):
        for c in range(ih):
            idx[r*W+c] = c*W + r
    return idx

def build_anti_transpose_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(iw):
        for c in range(ih):
            sr = ih - 1 - c; sc = iw - 1 - r
            if 0 <= sr < H and 0 <= sc < W:
                idx[r*W+c] = sr*W + sc
    return idx

def build_crop_gather(r0, c0, oh_, ow_):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(oh_):
        for c in range(ow_):
            sr = r0 + r; sc = c0 + c
            if sr < H and sc < W:
                idx[r*W+c] = sr*W + sc
    return idx

def build_tile_gather(tr, tc, ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(min(H, ih*tr)):
        for c in range(min(W, iw*tc)):
            idx[r*W+c] = (r % ih) * W + (c % iw)
    return idx

def build_scale_gather(factor, ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(min(H, ih*factor)):
        for c in range(min(W, iw*factor)):
            idx[r*W+c] = (r//factor)*W + (c//factor)
    return idx

def build_nonuniform_scale_gather(yf, xf, ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(min(H, ih*yf)):
        for c in range(min(W, iw*xf)):
            idx[r*W+c] = (r//yf)*W + (c//xf)
    return idx

def build_mirror_h_concat_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(2*iw):
            if c < W:
                src_c = c if c < iw else (2*iw-1-c)
                idx[r*W+c] = r*W + src_c
    return idx

def build_mirror_v_concat_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(2*ih):
        for c in range(iw):
            if r < H:
                src_r = r if r < ih else (2*ih-1-r)
                idx[r*W+c] = src_r*W + c
    return idx

def build_self_concat_h_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(2*iw):
            if c < W:
                idx[r*W+c] = r*W + (c % iw)
    return idx

def build_self_concat_v_gather(ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(2*ih):
        for c in range(iw):
            if r < H:
                idx[r*W+c] = (r % ih)*W + c
    return idx

def build_quad_mirror_gather(ih, iw, corner="tl"):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    if corner == "tl":
        for r in range(min(H, 2*ih)):
            for c in range(min(W, 2*iw)):
                sr = r if r < ih else (2*ih-1-r)
                sc = c if c < iw else (2*iw-1-c)
                idx[r*W+c] = sr*W + sc
    else:
        for r in range(min(H, 2*ih)):
            for c in range(min(W, 2*iw)):
                sr = (r - ih) if r >= ih else (ih - 1 - r)
                sc = (c - iw) if c >= iw else (iw - 1 - c)
                idx[r*W+c] = sr*W + sc
    return idx

def build_h_sym_complete_gathers(ih, iw):
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r*W+c] = r*W + c
            idx_m[r*W+c] = r*W + (iw-1-c)
    return [idx_id, idx_m]

def build_v_sym_complete_gathers(ih, iw):
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r*W+c] = r*W + c
            idx_m[r*W+c] = (ih-1-r)*W + c
    return [idx_id, idx_m]

def build_both_sym_complete_gathers(ih, iw):
    idxs = []
    for t in range(4):
        idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
        for r in range(ih):
            for c in range(iw):
                if t == 0: sr, sc = r, c
                elif t == 1: sr, sc = r, iw-1-c
                elif t == 2: sr, sc = ih-1-r, c
                else: sr, sc = ih-1-r, iw-1-c
                idx[r*W+c] = sr*W + sc
        idxs.append(idx)
    return idxs

def build_diag_sym_complete_gathers(ih, iw):
    if ih != iw: return None
    idx_id = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    idx_m = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_id[r*W+c] = r*W + c
            idx_m[r*W+c] = c*W + r
    return [idx_id, idx_m]

def build_all4_sym_complete_gathers(ih, iw):
    if ih != iw: return None
    idxs = build_both_sym_complete_gathers(ih, iw)
    idx_t = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    for r in range(ih):
        for c in range(iw):
            idx_t[r*W+c] = c*W + r
    idxs.append(idx_t)
    return idxs

# ============================================================
# V9 NEW DETECTORS + GATHER BUILDERS
# ============================================================

def _connected_components(g, bg=0):
    """4-connectivity labeling of non-bg cells. Returns list of (coords, color)."""
    h, w = g.shape
    visited = np.zeros_like(g, dtype=bool)
    comps = []
    for r in range(h):
        for c in range(w):
            if visited[r, c] or g[r, c] == bg:
                continue
            color = g[r, c]
            stack = [(r, c)]
            cells = []
            while stack:
                rr, cc = stack.pop()
                if not (0 <= rr < h and 0 <= cc < w): continue
                if visited[rr, cc] or g[rr, cc] != color: continue
                visited[rr, cc] = True
                cells.append((rr, cc))
                stack.extend([(rr+1, cc), (rr-1, cc), (rr, cc+1), (rr, cc-1)])
            comps.append((cells, color))
    return comps


def _connected_components_nonbg(g, bg=0):
    """4-connectivity labeling treating all non-bg as one group (color-agnostic)."""
    h, w = g.shape
    visited = np.zeros_like(g, dtype=bool)
    comps = []
    for r in range(h):
        for c in range(w):
            if visited[r, c] or g[r, c] == bg:
                continue
            stack = [(r, c)]
            cells = []
            while stack:
                rr, cc = stack.pop()
                if not (0 <= rr < h and 0 <= cc < w): continue
                if visited[rr, cc] or g[rr, cc] == bg: continue
                visited[rr, cc] = True
                cells.append((rr, cc))
                stack.extend([(rr+1, cc), (rr-1, cc), (rr, cc+1), (rr, cc-1)])
            comps.append(cells)
    return comps


# ---- Gravity (4 directions) ----
def _apply_gravity(g, direction, bg=0):
    """Shift all non-bg cells toward an edge, keeping column/row ordering."""
    out = np.full_like(g, bg)
    h, w = g.shape
    if direction == "down":
        for c in range(w):
            col = [g[r, c] for r in range(h) if g[r, c] != bg]
            for i, v in enumerate(col):
                out[h - len(col) + i, c] = v
    elif direction == "up":
        for c in range(w):
            col = [g[r, c] for r in range(h) if g[r, c] != bg]
            for i, v in enumerate(col):
                out[i, c] = v
    elif direction == "right":
        for r in range(h):
            row = [g[r, c] for c in range(w) if g[r, c] != bg]
            for i, v in enumerate(row):
                out[r, w - len(row) + i] = v
    elif direction == "left":
        for r in range(h):
            row = [g[r, c] for c in range(w) if g[r, c] != bg]
            for i, v in enumerate(row):
                out[r, i] = v
    return out


def detect_gravity(pairs):
    """Detect 4-direction gravity."""
    if not all(shapes_match(p) for p in pairs): return None
    for direction in ["down", "up", "left", "right"]:
        ok = True
        for p in pairs:
            ig = np.array(p["input"]); og = np.array(p["output"])
            if not np.array_equal(_apply_gravity(ig, direction), og):
                ok = False; break
        if ok: return direction
    return None


def build_gravity_gather(direction, h, w, sample_in, bg=0):
    """For each output pos, find the input src position (via simulation on sample).
    Note: gravity depends on input content, so gather-based gravity only works
    when the non-bg pattern is the same across pairs. Not general."""
    return None  # gravity is content-dependent; no pure gather solution.


# ---- Outline / shape-border extraction ----
def detect_outline_only(pairs):
    """Output = input with interior filled-in cells (surrounded on all 4 sides
    by same color) set to bg. Keeps just the outline."""
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        h, w = ig.shape
        exp = ig.copy()
        for r in range(h):
            for c in range(w):
                v = ig[r, c]
                if v == 0: continue
                if r == 0 or r == h - 1 or c == 0 or c == w - 1:
                    continue  # border of grid is always kept
                if (ig[r-1, c] == v and ig[r+1, c] == v and
                    ig[r, c-1] == v and ig[r, c+1] == v):
                    exp[r, c] = 0
        if not np.array_equal(exp, og): return False
    return True


# ---- Largest connected object crop ----
def detect_largest_object_bbox_crop(pairs):
    """Output = tight bbox-crop around the largest connected non-bg component."""
    if not pairs: return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        comps = _connected_components_nonbg(ig)
        if not comps: return False
        biggest = max(comps, key=lambda c: len(c))
        rs = [r for r, _ in biggest]; cs = [c for _, c in biggest]
        r0, r1 = min(rs), max(rs); c0, c1 = min(cs), max(cs)
        crop = ig[r0:r1+1, c0:c1+1]
        if not np.array_equal(crop, og): return False
    return True


def build_largest_bbox_gather(p):
    """Not pure-gather; depends on runtime content. Returns crop offsets only."""
    ig = np.array(p["input"])
    comps = _connected_components_nonbg(ig)
    if not comps: return None
    biggest = max(comps, key=lambda c: len(c))
    rs = [r for r, _ in biggest]; cs = [c for _, c in biggest]
    return (min(rs), min(cs), max(rs)-min(rs)+1, max(cs)-min(cs)+1)


def detect_fixed_largest_bbox_crop(pairs):
    """When every training pair has same bbox offset/size → use crop directly."""
    boxes = []
    for p in pairs:
        ig = np.array(p["input"])
        comps = _connected_components_nonbg(ig)
        if not comps: return None
        biggest = max(comps, key=lambda c: len(c))
        rs = [r for r, _ in biggest]; cs = [c for _, c in biggest]
        r0, r1 = min(rs), max(rs); c0, c1 = min(cs), max(cs)
        og = np.array(p["output"])
        if og.shape != (r1-r0+1, c1-c0+1): return None
        if not np.array_equal(ig[r0:r1+1, c0:c1+1], og): return None
        boxes.append((r0, c0, r1-r0+1, c1-c0+1))
    if len(set(boxes)) == 1:
        return boxes[0]
    return None


# ---- Smallest object crop ----
def detect_fixed_smallest_bbox_crop(pairs):
    """Like largest_bbox but smallest component — only returns when fixed."""
    boxes = []
    for p in pairs:
        ig = np.array(p["input"])
        comps = _connected_components_nonbg(ig)
        if not comps: return None
        smallest = min(comps, key=lambda c: len(c))
        rs = [r for r, _ in smallest]; cs = [c for _, c in smallest]
        r0, r1 = min(rs), max(rs); c0, c1 = min(cs), max(cs)
        og = np.array(p["output"])
        if og.shape != (r1-r0+1, c1-c0+1): return None
        if not np.array_equal(ig[r0:r1+1, c0:c1+1], og): return None
        boxes.append((r0, c0, r1-r0+1, c1-c0+1))
    if len(set(boxes)) == 1:
        return boxes[0]
    return None


# ---- Non-background bounding box crop (unique-color case) ----
def detect_fixed_nonbg_bbox(pairs):
    """Tight bbox around all non-bg pixels. Returns fixed offset if same across pairs."""
    boxes = []
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        mask = ig != 0
        if not mask.any(): return None
        rs, cs = np.where(mask)
        r0, r1 = rs.min(), rs.max(); c0, c1 = cs.min(), cs.max()
        crop = ig[r0:r1+1, c0:c1+1]
        if crop.shape != og.shape or not np.array_equal(crop, og): return None
        boxes.append((r0, c0, r1-r0+1, c1-c0+1))
    if len(set(boxes)) == 1:
        return boxes[0]
    return None


# ---- Repeating 1D pattern completion ----
def detect_repeat_1d_row(pairs):
    """Each output row is the input row extended by its period (covering bg cells)."""
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        h, w = ig.shape
        for r in range(h):
            nonbg = [(c, ig[r, c]) for c in range(w) if ig[r, c] != 0]
            if not nonbg: continue
            # try every period 1..w
            found = False
            for period in range(1, w + 1):
                filled = np.zeros(w, dtype=int)
                for c, v in nonbg:
                    base = c % period
                    # propagate v to all positions with same base
                    for cc in range(base, w, period):
                        if filled[cc] != 0 and filled[cc] != v:
                            break
                        filled[cc] = v
                    else:
                        continue
                    break
                else:
                    if np.array_equal(filled, og[r]):
                        found = True; break
            if not found:
                return False
    return True


# ---- Cross / plus marker at unique-color cell ----
def detect_cross_from_marker(pairs):
    """For each non-bg cell, draw a vertical+horizontal line through it of same color."""
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        h, w = ig.shape
        exp = np.zeros_like(ig)
        for r in range(h):
            for c in range(w):
                if ig[r, c] != 0:
                    v = ig[r, c]
                    exp[r, :] = np.where(exp[r, :] != 0, exp[r, :], v)
                    exp[:, c] = np.where(exp[:, c] != 0, exp[:, c], v)
        if not np.array_equal(exp, og): return False
    return True


# ---- Row/col constant stripe ----
def detect_every_other_row(pairs):
    """Output keeps every kth row (or column). Only detect k=2 even/odd cases."""
    if not pairs: return None
    for axis in [0, 1]:
        for start in [0, 1]:
            ok = True
            for p in pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                if axis == 0:
                    if og.shape != ((ig.shape[0] + (1 - start)) // 2, ig.shape[1]):
                        ok = False; break
                    if not np.array_equal(ig[start::2, :], og):
                        ok = False; break
                else:
                    if og.shape != (ig.shape[0], (ig.shape[1] + (1 - start)) // 2):
                        ok = False; break
                    if not np.array_equal(ig[:, start::2], og):
                        ok = False; break
            if ok: return (axis, start)
    return None


def build_every_other_gather(axis, start, ih, iw):
    idx = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
    if axis == 0:
        for r in range((ih - start + 1) // 2):
            for c in range(iw):
                idx[r*W+c] = (start + 2*r) * W + c
    else:
        for r in range(ih):
            for c in range((iw - start + 1) // 2):
                idx[r*W+c] = r * W + (start + 2*c)
    return idx


# ---- Fixed 3×3 spatial pattern kernel (conv-style) ----
def detect_dilate(pairs):
    """Output = 3×3 max dilation per channel (non-bg cells spread to 8 neighbors)."""
    if not all(shapes_match(p) for p in pairs): return False
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        h, w = ig.shape
        exp = np.zeros_like(ig)
        for r in range(h):
            for c in range(w):
                v = ig[r, c]
                if v == 0: continue
                for dr in range(-1, 2):
                    for dc in range(-1, 2):
                        rr, cc = r+dr, c+dc
                        if 0 <= rr < h and 0 <= cc < w:
                            if exp[rr, cc] == 0 or exp[rr, cc] == v:
                                exp[rr, cc] = v
        if not np.array_equal(exp, og): return False
    return True


def make_dilate_onnx_3x3():
    """Max-pool style dilation via MaxPool on each channel."""
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    node = oh.make_node("MaxPool", ["input"], ["output"],
                        kernel_shape=[3, 3], pads=[1, 1, 1, 1], strides=[1, 1])
    graph = oh.make_graph([node], "dilate", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


# ---- Fill enclosed region ----
def detect_fill_enclosed(pairs):
    """Output = input but with interior of closed shapes filled with a color."""
    if not all(shapes_match(p) for p in pairs): return None
    fill_color = None
    shape_color = None
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        diff = (ig != og)
        if not diff.any(): continue
        # Locations that changed must have been bg (0) in input
        if not np.all(ig[diff] == 0): return None
        fill_candidates = set(int(v) for v in og[diff].tolist())
        if len(fill_candidates) != 1: return None
        fc = fill_candidates.pop()
        if fill_color is None: fill_color = fc
        elif fill_color != fc: return None
        # The shape: non-bg color in input (expect a single shape-color)
        non_bg = set(int(v) for v in ig[ig != 0].tolist())
        if len(non_bg) != 1: return None
        sc = non_bg.pop()
        if shape_color is None: shape_color = sc
        elif shape_color != sc: return None
    if fill_color is None: return None
    # Verify: flood fill from outside → cells not reached from outside are "inside"
    for p in pairs:
        ig = np.array(p["input"]); og = np.array(p["output"])
        h, w = ig.shape
        reached = np.zeros_like(ig, dtype=bool)
        from collections import deque
        dq = deque()
        for c in range(w):
            if ig[0, c] == 0 and not reached[0, c]: dq.append((0, c)); reached[0, c] = True
            if ig[h-1, c] == 0 and not reached[h-1, c]: dq.append((h-1, c)); reached[h-1, c] = True
        for r in range(h):
            if ig[r, 0] == 0 and not reached[r, 0]: dq.append((r, 0)); reached[r, 0] = True
            if ig[r, w-1] == 0 and not reached[r, w-1]: dq.append((r, w-1)); reached[r, w-1] = True
        while dq:
            r, c = dq.popleft()
            for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
                rr, cc = r+dr, c+dc
                if 0<=rr<h and 0<=cc<w and not reached[rr, cc] and ig[rr, cc] == 0:
                    reached[rr, cc] = True; dq.append((rr, cc))
        exp = ig.copy()
        for r in range(h):
            for c in range(w):
                if ig[r, c] == 0 and not reached[r, c]:
                    exp[r, c] = fill_color
        if not np.array_equal(exp, og): return None
    return (shape_color, fill_color)


# ---- Gravity extensions for uniform-size input (gather-able) ----
# (Skipped — gravity is content-dependent; use conv-based learned approach.)

# ============================================================
# V9 COMPOSITION BUILDERS
# ============================================================

def compose_gathers(g1, g2):
    """Compose two spatial gathers: out[i] = in[g1[g2[i]]].
    g2 is applied first, then g1. But since they operate on the input via
    `input[out_idx] = input[gather[out_idx]]`, composing means:
    combined_gather[i] = g1[g2[i]] — i.e., first take g2's preimage,
    then g1's.
    Watch out for SAFE_PAD_IDX: if g2[i] is pad, so is combined.
    """
    combined = np.full_like(g2, SAFE_PAD_IDX)
    for i in range(len(g2)):
        a = g2[i]
        if a == SAFE_PAD_IDX:
            continue
        b = g1[a] if 0 <= a < len(g1) else SAFE_PAD_IDX
        combined[i] = b
    return combined


def make_gather_and_channel_gather_onnx(spatial_idx, channel_idx):
    """Thin wrapper: spatial gather + channel gather, picks up color compositions."""
    return make_spatial_then_channel_gather_onnx(spatial_idx, channel_idx)


# ============================================================
# WEIGHT BUILDERS
# ============================================================
def can_use_channel_gather_for_cmap(cmap, train_pairs):
    """Check if channel-gather is SAFE for this color map on this training data.

    Safe conditions:
    - Bijective (swap/cycle): every src is also some dst.
    - OR: for each non-identity src→dst, no training input contains value `dst`
      (so there's no ambiguity between "replace" and "swap").

    Returns True if safe, False if ambiguous (should use conv1x1 instead).
    """
    if cmap is None: return False
    changed = {s: d for s, d in cmap.items() if s != d}
    if not changed: return False

    # Case 1: fully bijective - always safe
    if set(changed.keys()) == set(changed.values()):
        return True

    # Case 2: non-bijective. We could use channel-gather IF the destinations
    # don't appear in any input (so our swap-like behavior is equivalent to replace).
    # But this is fragile - private data might contain the destination color.
    # Return False to prefer conv1x1.
    return False

def build_channel_gather_indices(cmap):
    """For a bijective color_map, build channel indices such that
    output_channel[dst] = input_channel[src] for each mapping."""
    gather_ch = np.arange(C, dtype=np.int32)
    for src, dst in cmap.items():
        if src != dst:
            gather_ch[dst] = src
    return gather_ch

def build_color_map_weight(color_map):
    weight = np.zeros((C, C, 1, 1), dtype=np.float32)
    for src, dst in color_map.items():
        if 0 <= src < C and 0 <= dst < C:
            weight[dst, src, 0, 0] = 1.0
    for ch in range(C):
        if ch not in color_map:
            weight[ch, ch, 0, 0] = 1.0
    return weight

def build_nonzero_recolor_weight(target_color):
    """1x1 conv: all non-bg channels → target; bg → bg."""
    weight = np.zeros((C, C, 1, 1), dtype=np.float32)
    weight[0, 0, 0, 0] = 1.0  # bg → bg
    for src in range(1, C):
        weight[target_color, src, 0, 0] = 1.0
    return weight

def build_bg_recolor_weight(target_color):
    """1x1 conv: bg → target; non-bg stay."""
    weight = np.zeros((C, C, 1, 1), dtype=np.float32)
    weight[target_color, 0, 0, 0] = 1.0
    for src in range(1, C):
        weight[src, src, 0, 0] = 1.0
    return weight

# ============================================================
# LEARNED CONV with data augmentation
# ============================================================

def augment_pairs(pairs):
    """Create rotation/flip augmented training pairs. Only safe when the
    transformation is rotation/flip equivariant — a strong assumption.
    We return just originals here; we don't know in general.
    """
    return pairs  # Conservative: no augmentation by default (breaks non-equivariant tasks)

def try_learned_conv_fast(pairs, all_pairs, kernel_size=1, max_steps=1500, lr=0.02):
    try:
        import torch, torch.nn as nn
    except ImportError:
        return None
    if not all(shapes_match(p) for p in pairs): return None

    inp = torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]), dtype=torch.float32)
    out = torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]), dtype=torch.float32)

    pad = kernel_size // 2
    conv = nn.Conv2d(C, C, kernel_size=kernel_size, padding=pad, bias=False)
    nn.init.zeros_(conv.weight)
    for i in range(C):
        conv.weight.data[i, i, pad, pad] = 1.0  # near-identity init

    optimizer = torch.optim.Adam(conv.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=500, gamma=0.5)
    best_loss = float("inf"); best_state = None; plateau = 0

    for step in range(max_steps):
        optimizer.zero_grad()
        pred = conv(inp)
        loss = nn.functional.mse_loss(pred, out)
        loss.backward(); optimizer.step(); scheduler.step()
        l = loss.item()
        if l < best_loss - 1e-7:
            best_loss = l; best_state = copy.deepcopy(conv.state_dict()); plateau = 0
        else:
            plateau += 1
        if step == 250 and best_loss > 0.3: return None
        if step == 600 and best_loss > 0.1: return None
        if plateau > 400 and best_loss > 0.01: return None
        if best_loss < 1e-6: break

    if best_loss > 0.005: return None
    conv.load_state_dict(best_state)
    weight = conv.weight.detach().numpy()

    # Prune near-zero weights
    weight_pruned = np.where(np.abs(weight) < 0.01, 0.0, weight)
    # Also try rounding to integers
    weight_rounded = np.round(weight)

    # Try pruned, rounded, original — pick smallest cost that works
    best_model = None
    best_cost = float('inf')
    for w_try in [weight_rounded, weight_pruned, weight]:
        m = make_conv_onnx(w_try, kernel_size=kernel_size)
        if check_model_correct(m, all_pairs):
            c = estimate_model_cost(m)
            if c < best_cost:
                best_cost = c; best_model = m
    return best_model

def try_two_layer_conv_fast(pairs, all_pairs, ks1=3, ks2=1, hidden=16, max_steps=2500, lr=0.015):
    try:
        import torch, torch.nn as nn
    except ImportError:
        return None
    if not all(shapes_match(p) for p in pairs): return None

    inp = torch.tensor(np.stack([grid_to_tensor(p["input"])[0] for p in pairs]), dtype=torch.float32)
    out = torch.tensor(np.stack([grid_to_tensor(p["output"])[0] for p in pairs]), dtype=torch.float32)

    pad1 = ks1 // 2; pad2 = ks2 // 2
    torch.manual_seed(0)
    model_nn = nn.Sequential(
        nn.Conv2d(C, hidden, kernel_size=ks1, padding=pad1, bias=True),
        nn.ReLU(),
        nn.Conv2d(hidden, C, kernel_size=ks2, padding=pad2, bias=False),
    )

    optimizer = torch.optim.Adam(model_nn.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps)
    best_loss = float("inf"); best_state = None; plateau = 0

    for step in range(max_steps):
        optimizer.zero_grad()
        pred = model_nn(inp)
        loss = nn.functional.mse_loss(pred, out)
        loss.backward(); optimizer.step(); scheduler.step()
        l = loss.item()
        if l < best_loss - 1e-7:
            best_loss = l; best_state = copy.deepcopy(model_nn.state_dict()); plateau = 0
        else:
            plateau += 1
        if step == 300 and best_loss > 0.5: return None
        if step == 800 and best_loss > 0.15: return None
        if plateau > 500 and best_loss > 0.005: return None
        if best_loss < 1e-6: break

    if best_loss > 0.005: return None
    model_nn.load_state_dict(best_state)
    w1 = model_nn[0].weight.detach().numpy()
    b1 = model_nn[0].bias.detach().numpy()
    w2 = model_nn[2].weight.detach().numpy()

    # Prune
    w1 = np.where(np.abs(w1) < 0.01, 0.0, w1)
    w2 = np.where(np.abs(w2) < 0.01, 0.0, w2)

    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    w1_t = onh.from_array(w1.astype(np.float32), name="w1")
    w1_n = oh.make_node("Constant", [], ["w1"], value=w1_t)
    b1_t = onh.from_array(b1.astype(np.float32), name="b1")
    b1_n = oh.make_node("Constant", [], ["b1"], value=b1_t)
    c1 = oh.make_node("Conv", ["input", "w1", "b1"], ["c1_out"],
                      kernel_shape=[ks1, ks1], pads=[pad1, pad1, pad1, pad1])
    relu = oh.make_node("Relu", ["c1_out"], ["relu"])
    w2_t = onh.from_array(w2.astype(np.float32), name="w2")
    w2_n = oh.make_node("Constant", [], ["w2"], value=w2_t)
    c2 = oh.make_node("Conv", ["relu", "w2"], ["output"],
                      kernel_shape=[ks2, ks2], pads=[pad2, pad2, pad2, pad2])
    nodes = [w1_n, b1_n, c1, relu, w2_n, c2]
    graph = oh.make_graph(nodes, "two_layer_conv", [X], [Y])
    m = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    m.ir_version = 8
    if check_model_correct(m, all_pairs):
        return m
    return None

# ============================================================
# V9 IMPROVED NEURAL TRAINERS (use arc-gen + deeper nets)
# ============================================================

def _build_train_tensors(train_pairs, arc_gen, max_aux=40):
    """Combine train + a subset of arc-gen into one tensor stack for training.
    Returns (inp_tensor, out_tensor) of shape (N, 10, 30, 30), or (None, None)
    if any pair has mismatched shape.
    """
    # Use only pairs where input/output have same shape (the typical case for
    # same-size detectors). For size-change tasks arc_gen may still match.
    pool = list(train_pairs)
    if arc_gen:
        rng = random.Random(17)
        ag = list(arc_gen); rng.shuffle(ag)
        pool = pool + ag[:max_aux]
    inputs = []; outputs = []
    for p in pool:
        try:
            inputs.append(grid_to_tensor(p["input"])[0])
            outputs.append(grid_to_tensor(p["output"])[0])
        except Exception:
            continue
    if not inputs:
        return None, None
    return np.stack(inputs), np.stack(outputs)


def try_learned_conv_plus(train_pairs, all_pairs, arc_gen,
                          kernel_size=1, max_steps=1200, lr=0.02,
                          max_aux=30, accept_loss=0.002):
    """Learned conv that additionally trains on arc-gen samples."""
    try:
        import torch, torch.nn as nn
    except ImportError:
        return None
    if not all(shapes_match(p) for p in train_pairs):
        return None

    inp_np, out_np = _build_train_tensors(train_pairs, arc_gen, max_aux=max_aux)
    if inp_np is None: return None
    inp = torch.tensor(inp_np, dtype=torch.float32)
    out = torch.tensor(out_np, dtype=torch.float32)

    pad = kernel_size // 2
    conv = nn.Conv2d(C, C, kernel_size=kernel_size, padding=pad, bias=False)
    nn.init.zeros_(conv.weight)
    for i in range(C):
        conv.weight.data[i, i, pad, pad] = 1.0  # near-identity init

    optimizer = torch.optim.Adam(conv.parameters(), lr=lr)
    best_loss = float("inf"); best_state = None; plateau = 0

    for step in range(max_steps):
        optimizer.zero_grad()
        pred = conv(inp)
        loss = nn.functional.mse_loss(pred, out)
        loss.backward(); optimizer.step()
        l = loss.item()
        if l < best_loss - 1e-7:
            best_loss = l; best_state = copy.deepcopy(conv.state_dict()); plateau = 0
        else:
            plateau += 1
        if step == 200 and best_loss > 0.35: return None
        if step == 500 and best_loss > 0.12: return None
        if plateau > 300 and best_loss > accept_loss: break
        if best_loss < 1e-6: break

    if best_loss > accept_loss: return None
    conv.load_state_dict(best_state)
    weight = conv.weight.detach().numpy()

    # Try multiple quantizations, pick cheapest working
    weight_pruned = np.where(np.abs(weight) < 0.01, 0.0, weight)
    weight_tiny_pruned = np.where(np.abs(weight) < 0.005, 0.0, weight)
    weight_rounded = np.round(weight)
    weight_rounded_half = np.round(weight * 2) / 2
    best_model = None; best_cost = float('inf')
    for w_try in [weight_rounded, weight_rounded_half, weight_tiny_pruned, weight_pruned, weight]:
        m = make_conv_onnx(w_try, kernel_size=kernel_size)
        if check_model_correct(m, all_pairs):
            c = estimate_model_cost(m)
            if c < best_cost: best_cost = c; best_model = m
    return best_model


def _make_deep_conv_onnx(weights, biases, kernel_sizes, has_relu):
    """Make a multi-layer conv ONNX network.
    weights: list of numpy arrays.
    biases: list of numpy arrays or None per layer.
    kernel_sizes: list of ints.
    has_relu: list of bools, whether to apply ReLU after each conv (except last).
    """
    X = oh.make_tensor_value_info("input", TensorProto.FLOAT, [1, C, H, W])
    Y = oh.make_tensor_value_info("output", TensorProto.FLOAT, [1, C, H, W])
    nodes = []
    prev = "input"
    for i, (w, b, ks) in enumerate(zip(weights, biases, kernel_sizes)):
        pad = ks // 2
        w_name = f"w{i}"; b_name = f"b{i}"
        w_tensor = onh.from_array(w.astype(np.float32), name=w_name)
        nodes.append(oh.make_node("Constant", [], [w_name], value=w_tensor))
        conv_inputs = [prev, w_name]
        if b is not None:
            b_tensor = onh.from_array(b.astype(np.float32), name=b_name)
            nodes.append(oh.make_node("Constant", [], [b_name], value=b_tensor))
            conv_inputs.append(b_name)
        out_name = "output" if (i == len(weights) - 1 and not has_relu[i]) else f"c{i}"
        nodes.append(oh.make_node("Conv", conv_inputs, [out_name],
                                   kernel_shape=[ks, ks],
                                   pads=[pad, pad, pad, pad]))
        prev = out_name
        if has_relu[i]:
            relu_out = "output" if i == len(weights) - 1 else f"r{i}"
            nodes.append(oh.make_node("Relu", [prev], [relu_out]))
            prev = relu_out
    graph = oh.make_graph(nodes, "deep_conv", [X], [Y])
    model = oh.make_model(graph, opset_imports=[oh.make_opsetid("", 17)])
    model.ir_version = 8
    return model


def try_deep_conv(train_pairs, all_pairs, arc_gen,
                  kernel_sizes=(3, 1), hiddens=(16,), max_steps=2500,
                  lr=0.01, max_aux=30, accept_loss=0.004, dilations=None):
    """Generic deep conv trainer: multi-layer CNN with ReLU between.
    kernel_sizes: list of kernel sizes, one per layer (must be length = len(hiddens) + 1).
    hiddens: list of hidden channel counts (not counting C in/out).
    """
    try:
        import torch, torch.nn as nn
    except ImportError:
        return None
    if not all(shapes_match(p) for p in train_pairs): return None
    inp_np, out_np = _build_train_tensors(train_pairs, arc_gen, max_aux=max_aux)
    if inp_np is None: return None
    inp = torch.tensor(inp_np, dtype=torch.float32)
    out = torch.tensor(out_np, dtype=torch.float32)

    layers = []
    prev_ch = C
    channels = list(hiddens) + [C]  # last = C output
    if dilations is None:
        dilations = [1] * len(kernel_sizes)
    for i, (ks, ch) in enumerate(zip(kernel_sizes, channels)):
        dil = dilations[i]
        pad = (ks // 2) * dil
        layers.append(nn.Conv2d(prev_ch, ch, kernel_size=ks, padding=pad,
                                 dilation=dil, bias=(i < len(kernel_sizes) - 1)))
        if i < len(kernel_sizes) - 1:
            layers.append(nn.ReLU())
        prev_ch = ch
    torch.manual_seed(0)
    model_nn = nn.Sequential(*layers)

    optimizer = torch.optim.Adam(model_nn.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max_steps)
    best_loss = float("inf"); best_state = None; plateau = 0
    for step in range(max_steps):
        optimizer.zero_grad()
        pred = model_nn(inp)
        loss = nn.functional.mse_loss(pred, out)
        loss.backward(); optimizer.step(); scheduler.step()
        l = loss.item()
        if l < best_loss - 1e-7:
            best_loss = l; best_state = copy.deepcopy(model_nn.state_dict()); plateau = 0
        else:
            plateau += 1
        if step == 250 and best_loss > 0.6: return None
        if step == 700 and best_loss > 0.2: return None
        if plateau > 500 and best_loss > accept_loss: break
        if best_loss < 1e-6: break

    if best_loss > accept_loss: return None
    model_nn.load_state_dict(best_state)

    # Extract weights
    weights = []; biases = []
    conv_idx = 0
    for m in model_nn:
        if isinstance(m, nn.Conv2d):
            w = m.weight.detach().numpy()
            b = m.bias.detach().numpy() if m.bias is not None else None
            weights.append(w); biases.append(b); conv_idx += 1
    has_relu = [True] * (len(weights) - 1) + [False]

    # Dilations aren't directly supported by our simple _make_deep_conv_onnx helper.
    # For now only build when all dilations are 1.
    if any(d != 1 for d in dilations):
        return None

    # Try pruning levels
    best_model = None; best_cost = float('inf')
    for thresh in [0.0, 0.003, 0.01, 0.03]:
        w_try = [np.where(np.abs(w) < thresh, 0.0, w) for w in weights]
        m = _make_deep_conv_onnx(w_try, biases, list(kernel_sizes), has_relu)
        if check_model_correct(m, all_pairs):
            c = estimate_model_cost(m)
            if c < best_cost: best_cost = c; best_model = m
    return best_model


def try_always_attempt_fallback(train_pairs, all_pairs, arc_gen, max_aux=40):
    """Last-resort: try a sequence of increasingly expensive networks.
    Returns the FIRST correct model found (may not be minimum cost), accepting
    higher training loss. Used when everything cheaper has failed so any points
    are better than zero.
    """
    configs = [
        dict(kernel_sizes=(3, 1),      hiddens=(16,),       max_steps=2500, lr=0.012, accept_loss=0.005),
        dict(kernel_sizes=(3, 3),      hiddens=(16,),       max_steps=2500, lr=0.012, accept_loss=0.005),
        dict(kernel_sizes=(3, 3, 1),   hiddens=(16, 16),    max_steps=3000, lr=0.01,  accept_loss=0.006),
        dict(kernel_sizes=(5, 1),      hiddens=(12,),       max_steps=2000, lr=0.012, accept_loss=0.006),
        dict(kernel_sizes=(3, 3, 3),   hiddens=(20, 20),    max_steps=3500, lr=0.01,  accept_loss=0.008),
        dict(kernel_sizes=(3, 1, 3, 1),hiddens=(24, 24, 16),max_steps=4000, lr=0.008, accept_loss=0.01),
        dict(kernel_sizes=(5, 3, 1),   hiddens=(24, 16),    max_steps=3000, lr=0.01,  accept_loss=0.01),
    ]
    for cfg in configs:
        try:
            m = try_deep_conv(train_pairs, all_pairs, arc_gen, max_aux=max_aux, **cfg)
        except Exception:
            continue
        if m is not None:
            return m
    return None


# ============================================================
# SOLVER
# ============================================================
def solve_task(task_data, task_name="", time_budget=25.0):
    train_pairs = task_data.get("train", [])
    test_pairs = task_data.get("test", [])
    arc_gen = task_data.get("arc-gen", [])

    if not train_pairs:
        return make_identity_onnx()

    # Validate on much more arc-gen data than v2 (V2 used 10, now 50+)
    # Shuffle arc-gen but use seed for determinism
    rng = random.Random(42)
    ag = list(arc_gen)
    rng.shuffle(ag)
    all_pairs = train_pairs + test_pairs + ag[:MAX_ARC_GEN_VALIDATE]

    info = analyze_transformation(train_pairs)
    best_model = None; best_cost = float('inf')
    t_start = time.time()

    def try_model(model, label=""):
        nonlocal best_model, best_cost
        if model is None: return False
        if check_model_correct(model, all_pairs):
            cost = estimate_model_cost(model)
            if cost < best_cost:
                best_cost = cost; best_model = model
            return True
        return False

    def elapsed(): return time.time() - t_start

    # ---- CHEAP CHECKS ----
    if info.get("identity"):
        if try_model(make_identity_onnx(), "identity"):
            return best_model

    # Color maps (global) - prefer channel-gather when safe.
    # IMPORTANT: If a spatial transform also fits on training, the data is
    # ambiguous (values rotated could look like a color permutation). In that case
    # we must prefer the spatial transform which generalizes to arbitrary data.
    has_spatial_same_size = (
        (detect_rotation(train_pairs) == 180) or  # only rot180 preserves size
        (detect_flip(train_pairs) is not None) or
        detect_transpose(train_pairs) or
        detect_anti_transpose(train_pairs)
    )
    # Also check: for different-shape outputs, other spatial transforms might disguise
    has_spatial_any = (
        detect_rotation(train_pairs) is not None or
        detect_flip(train_pairs) is not None or
        detect_transpose(train_pairs) or
        detect_anti_transpose(train_pairs)
    )
    if info.get("color_map") and info["same_size"]:
        cmap = info["color_map"]
        if can_use_channel_gather_for_cmap(cmap, train_pairs) and not has_spatial_same_size:
            gi = build_channel_gather_indices(cmap)
            try_model(make_channel_gather_onnx(gi), "color_map_ch")
        w = build_color_map_weight(cmap)
        try_model(make_conv1x1_onnx(w), "color_map_conv")

    # Nonzero recolor (silhouette)
    tc = detect_nonzero_recolor(train_pairs)
    if tc is not None:
        try_model(make_conv1x1_onnx(build_nonzero_recolor_weight(tc)), "nonzero_recolor")

    # BG recolor
    tc = detect_bg_recolor(train_pairs)
    if tc is not None:
        try_model(make_conv1x1_onnx(build_bg_recolor_weight(tc)), "bg_recolor")

    # Color replace / swap - prefer channel-gather for bijective (safe) cases
    # but NOT when spatial transforms also fit (ambiguity).
    repl = detect_color_replace(train_pairs)
    if repl:
        cmap = {i: i for i in range(C)}; cmap.update(repl)
        if can_use_channel_gather_for_cmap(cmap, train_pairs) and not has_spatial_same_size:
            gi = build_channel_gather_indices(cmap)
            try_model(make_channel_gather_onnx(gi), "color_replace_ch")
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), "color_replace_conv")

    sw = detect_color_swap(train_pairs)
    if sw and not has_spatial_same_size:
        # Swap is always bijective; with both-direction evidence from detect_color_swap
        # and no spatial transform, it's safe.
        a, b = sw
        cmap = {i: i for i in range(C)}; cmap[a] = b; cmap[b] = a
        gi = build_channel_gather_indices(cmap)
        try_model(make_channel_gather_onnx(gi), "color_swap_ch")
        try_model(make_conv1x1_onnx(build_color_map_weight(cmap)), "color_swap_conv")

    # Rotation (size-aware, maskless thanks to SAFE_PAD_IDX)
    rot = detect_rotation(train_pairs)
    if rot:
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            gi = build_rotation_gather(rot, ih, iw)
            try_model(make_gather_onnx(gi), f"rot_{rot}")
            # v7 CHEAP: try Transpose/Slice-based rotation (cost ~200 vs 4563)
            if rot == 90:
                try_model(make_cheap_rot90ccw_framed(ih, iw), f"rot_{rot}_cheap")
            elif rot == 180:
                try_model(make_cheap_rot180_framed(ih, iw), f"rot_{rot}_cheap")
            elif rot == 270:
                try_model(make_cheap_rot90cw_framed(ih, iw), f"rot_{rot}_cheap")

    flip = detect_flip(train_pairs)
    if flip:
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_flip_gather(flip, ih, iw)), f"flip_{flip}")
            # v7 CHEAP: Slice-based flip
            if flip == "vertical":
                try_model(make_cheap_flip_h_framed(ih, iw), f"flip_{flip}_cheap")
            elif flip == "horizontal":
                try_model(make_cheap_flip_w_framed(ih, iw), f"flip_{flip}_cheap")
            elif flip == "both":
                try_model(make_cheap_rot180_framed(ih, iw), f"flip_{flip}_cheap")

    if detect_transpose(train_pairs):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_transpose_gather(ih, iw)), "transpose")
            # v7 CHEAP: Transpose op
            try_model(make_cheap_transpose_framed(ih, iw), "transpose_cheap")
    if detect_anti_transpose(train_pairs):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_anti_transpose_gather(ih, iw)), "anti_transpose")
            # v7 CHEAP: Transpose + flip both axes
            try_model(make_cheap_anti_transpose_framed(ih, iw), "anti_transpose_cheap")

    # Crop / tile / scale
    crop = detect_crop(train_pairs)
    if crop:
        r0, c0, oh_, ow_ = crop
        try_model(make_gather_onnx(build_crop_gather(r0, c0, oh_, ow_)), "crop")
        # v7 CHEAP: Slice + Pad
        try_model(make_cheap_crop(r0, c0, oh_, ow_), "crop_cheap")

    tile = detect_tile(train_pairs)
    if tile:
        tr, tc = tile
        ih, iw = np.array(train_pairs[0]["input"]).shape
        oh_ = ih * tr; ow_ = iw * tc
        if oh_ <= H and ow_ <= W:
            try_model(make_gather_onnx(build_tile_gather(tr, tc, ih, iw)), "tile")
            # v7 CHEAP: Tile op
            try_model(make_cheap_tile(tr, tc, ih, iw), "tile_cheap")

    sc = detect_scale(train_pairs)
    if sc:
        ih, iw = np.array(train_pairs[0]["input"]).shape
        oh_ = ih * sc; ow_ = iw * sc
        if oh_ <= H and ow_ <= W:
            try_model(make_gather_onnx(build_scale_gather(sc, ih, iw)), f"scale_{sc}")
            # v7 CHEAP: Resize op
            try_model(make_cheap_upscale(sc, ih, iw), f"scale_{sc}_cheap")

    us = detect_upscale(train_pairs)
    if us:
        yf, xf = us
        ih, iw = np.array(train_pairs[0]["input"]).shape
        oh_ = ih * yf; ow_ = iw * xf
        if oh_ <= H and ow_ <= W:
            try_model(make_gather_onnx(build_nonuniform_scale_gather(yf, xf, ih, iw)),
                      f"upscale_{yf}x{xf}")
            # v7 CHEAP: when yf == xf, use Resize
            if yf == xf:
                try_model(make_cheap_upscale(yf, ih, iw), f"upscale_{yf}x{xf}_cheap")

    # Pixel permutation (conservative)
    perm = detect_pixel_permutation(train_pairs)
    if perm is not None:
        try_model(make_gather_onnx(perm), "pixel_perm")

    # Mirror patterns
    if detect_mirror_h_concat(train_pairs):
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_gather_onnx(build_mirror_h_concat_gather(ih, iw)), "mirror_h")
        # v7 CHEAP: Concat + Slice_reverse
        try_model(make_cheap_concat_flip_w(ih, iw), "mirror_h_cheap")
    if detect_mirror_v_concat(train_pairs):
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_gather_onnx(build_mirror_v_concat_gather(ih, iw)), "mirror_v")
        # v7 CHEAP: Concat + Slice_reverse
        try_model(make_cheap_concat_flip_h(ih, iw), "mirror_v_cheap")

    if detect_self_concat_h(train_pairs):
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_gather_onnx(build_self_concat_h_gather(ih, iw)), "self_h")
        # v7 CHEAP: Concat with itself
        try_model(make_cheap_self_concat_w(ih, iw), "self_h_cheap")
    if detect_self_concat_v(train_pairs):
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_gather_onnx(build_self_concat_v_gather(ih, iw)), "self_v")
        try_model(make_cheap_self_concat_h(ih, iw), "self_v_cheap")

    qm = detect_quad_mirror(train_pairs)
    if qm:
        ih, iw = np.array(train_pairs[0]["input"]).shape
        try_model(make_gather_onnx(build_quad_mirror_gather(ih, iw, qm)), "quad_mirror")
        # v7 CHEAP: 4-way concat mirror
        if qm == "tl":  # top-left placement (standard quadrant mirror)
            try_model(make_cheap_quadrant_mirror(ih, iw), "quad_mirror_cheap")

    # Composite: spatial transform + color map.
    if rot and info.get("color_map"):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            gi = build_rotation_gather(rot, ih, iw)
            cmap = info["color_map"]
            if can_use_channel_gather_for_cmap(cmap, train_pairs):
                ch_gi = build_channel_gather_indices(cmap)
                try_model(make_spatial_then_channel_gather_onnx(gi, ch_gi), "rot_ch_colormap")
            w = build_color_map_weight(cmap)
            try_model(make_gather_then_conv1x1_onnx(gi, w), "rot_colormap")
    if flip and info.get("color_map"):
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            gi = build_flip_gather(flip, ih, iw)
            cmap = info["color_map"]
            if can_use_channel_gather_for_cmap(cmap, train_pairs):
                ch_gi = build_channel_gather_indices(cmap)
                try_model(make_spatial_then_channel_gather_onnx(gi, ch_gi), "flip_ch_colormap")
            w = build_color_map_weight(cmap)
            try_model(make_gather_then_conv1x1_onnx(gi, w), "flip_colormap")

    # Grid-of-grids
    gog = detect_grid_of_grids_fixed(train_pairs)
    if gog:
        br, bc, oh_, ow_ = gog
        r0 = br * oh_; c0 = bc * ow_
        try_model(make_gather_onnx(build_crop_gather(r0, c0, oh_, ow_)), "grid_select")

    # Symmetry completion
    if info["same_size"] and info["all_same_in_size"]:
        ih, iw = np.array(train_pairs[0]["input"]).shape
        if detect_h_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_h_sym_complete_gathers(ih, iw)), "h_sym")
        if detect_v_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_v_sym_complete_gathers(ih, iw)), "v_sym")
        if detect_both_symmetry_complete(train_pairs):
            try_model(make_symmetry_completion_onnx(build_both_sym_complete_gathers(ih, iw)), "both_sym")
        if ih == iw:
            if detect_diag_symmetry_complete(train_pairs):
                g = build_diag_sym_complete_gathers(ih, iw)
                if g is not None:
                    try_model(make_symmetry_completion_onnx(g), "diag_sym")
            if detect_all4_symmetry_complete(train_pairs):
                g = build_all4_sym_complete_gathers(ih, iw)
                if g is not None:
                    try_model(make_symmetry_completion_onnx(g), "all4_sym")

    # ---- V8 NEW DETECTORS ----

    # Translation (Slice+Pad, 0 MACs)
    trans = detect_translation(train_pairs)
    if trans is not None:
        dr, dc = trans
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            m = make_cheap_framed_translation(ih, iw, dr, dc)
            if m is not None:
                try_model(m, f"translate_{dr}_{dc}")

    # Downscale
    in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
    if len(in_shapes) == 1:
        ih, iw = list(in_shapes)[0]
        ds = detect_downscale(train_pairs)
        if ds is not None:
            try_model(make_cheap_downscale(ds, ih, iw), f"downscale_{ds}")

    # Color filter (set of colors -> 0)
    cf = detect_color_filter(train_pairs)
    if cf is not None and info["same_size"]:
        cmap = {c_: 0 for c_ in cf}
        cmap.update({c_: c_ for c_ in range(C) if c_ not in cf})
        if can_use_channel_gather_for_cmap(cmap, train_pairs):
            ch_gi = build_channel_gather_indices(cmap)
            try_model(make_channel_gather_onnx(ch_gi), "color_filter_ch")

    # Row/col broadcast
    in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
    if len(in_shapes) == 1:
        ih, iw = list(in_shapes)[0]
        rb = detect_row_broadcast(train_pairs)
        if rb is not None:
            try_model(make_cheap_row_broadcast(ih, iw, rb), f"row_broadcast_{rb}")
        cb = detect_col_broadcast(train_pairs)
        if cb is not None:
            try_model(make_cheap_col_broadcast(ih, iw, cb), f"col_broadcast_{cb}")

    # Border add
    in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
    if len(in_shapes) == 1:
        ih, iw = list(in_shapes)[0]
        ba = detect_border_add(train_pairs)
        if ba is not None:
            b, fill = ba
            if ih + 2*b <= H and iw + 2*b <= W:
                m = make_cheap_border_add(ih, iw, b, fill)
                if m is not None:
                    try_model(m, f"border_{b}_{fill}")

    # Crop + color
    ccp = detect_crop_plus_color(train_pairs)
    if ccp is not None:
        r0, c0, oh_, ow_, cmap = ccp
        try_model(make_cheap_crop_then_color(r0, c0, oh_, ow_, cmap), "crop_color")

    # Concat rot180 w
    in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
    if len(in_shapes) == 1:
        ih, iw = list(in_shapes)[0]
        if detect_concat_rot180_w(train_pairs) and ih <= H and iw * 2 <= W:
            try_model(make_cheap_concat_rot180_w(ih, iw), "concat_rot180_w")

    # Spatial zero-ambiguity (catches arbitrary pixel permutations)
    m = try_spatial_zero_ambiguity(all_pairs)
    if m is not None:
        try_model(m, "spatial_zero_ambig")

    # ---- V9 NEW DETECTORS ----

    # Dilation (3x3 MaxPool)
    if detect_dilate(train_pairs):
        try_model(make_dilate_onnx_3x3(), "dilate_3x3")

    # Outline (content-dependent; only solvable by learning, not pure gather)
    # Skipped here, CNN fallback handles it.

    # Largest/smallest object fixed bbox crop — only safe when offset is fixed
    lob = detect_fixed_largest_bbox_crop(train_pairs)
    if lob is not None:
        r0, c0, oh_, ow_ = lob
        try_model(make_cheap_crop(r0, c0, oh_, ow_), "largest_bbox_crop")

    sob = detect_fixed_smallest_bbox_crop(train_pairs)
    if sob is not None:
        r0, c0, oh_, ow_ = sob
        try_model(make_cheap_crop(r0, c0, oh_, ow_), "smallest_bbox_crop")

    nbb = detect_fixed_nonbg_bbox(train_pairs)
    if nbb is not None:
        r0, c0, oh_, ow_ = nbb
        try_model(make_cheap_crop(r0, c0, oh_, ow_), "nonbg_bbox_crop")

    # Every-other-row / every-other-col
    eo = detect_every_other_row(train_pairs)
    if eo is not None:
        axis, start = eo
        in_shapes = set(np.array(p["input"]).shape for p in train_pairs)
        if len(in_shapes) == 1:
            ih, iw = list(in_shapes)[0]
            try_model(make_gather_onnx(build_every_other_gather(axis, start, ih, iw)),
                      f"every_other_{axis}_{start}")

    # ---- V9 COMPOSITION SEARCH ----
    # 2-step: spatial transform + color map
    in_shapes_set = set(np.array(p["input"]).shape for p in train_pairs)
    same_in_shape = len(in_shapes_set) == 1
    cmap_info = info.get("color_map") if info.get("same_size") else None

    # Compose rot/flip/transpose/anti-transpose/crop/tile/scale/downscale with color map
    def _try_spatial_plus_color(spatial_gi, label):
        """Given spatial gather, try both (gather + color map) via channel gather
        or conv1x1 composition. Derives the color map implicitly from pairs."""
        try:
            cmap_accum = None
            for p in train_pairs:
                ig = np.array(p["input"]); og = np.array(p["output"])
                ih2, iw2 = ig.shape
                if ih2 > H or iw2 > W: return None
                flat = np.zeros((H, W), dtype=np.int32)
                flat[:ih2, :iw2] = ig
                flat_full = flat.flatten()
                mapped = np.zeros(HW, dtype=np.int32)
                for i in range(HW):
                    si = int(spatial_gi[i])
                    mapped[i] = flat_full[si] if si != SAFE_PAD_IDX else 0
                oh2, ow2 = og.shape
                if oh2 > H or ow2 > W: return None
                region = mapped.reshape(H, W)[:oh2, :ow2]
                if region.shape != og.shape: return None
                cmap_local = {}
                for a, b in zip(region.flatten(), og.flatten()):
                    a, b = int(a), int(b)
                    if a in cmap_local and cmap_local[a] != b:
                        return None
                    cmap_local[a] = b
                if cmap_accum is None:
                    cmap_accum = dict(cmap_local)
                else:
                    for k, v in cmap_local.items():
                        if k in cmap_accum and cmap_accum[k] != v:
                            return None
                        cmap_accum[k] = v
            if cmap_accum is None:
                return None
            cmap_full = {i: i for i in range(C)}
            cmap_full.update(cmap_accum)
            if all(k == v for k, v in cmap_full.items()):
                return None  # identity color map — pure spatial covers this
            if can_use_channel_gather_for_cmap(cmap_full, train_pairs):
                ch_gi = build_channel_gather_indices(cmap_full)
                try_model(make_spatial_then_channel_gather_onnx(spatial_gi, ch_gi),
                          f"{label}_ch_color")
            w = build_color_map_weight(cmap_full)
            try_model(make_gather_then_conv1x1_onnx(spatial_gi, w), f"{label}_conv_color")
        except Exception:
            return None

    if same_in_shape:
        ih, iw = list(in_shapes_set)[0]
        # transpose + color
        if detect_transpose(train_pairs) is False:
            # try it anyway if shapes allow — skip since we already handled pure transpose
            pass
        # Compose transpose with hypothetical color map
        if ih <= W and iw <= H:
            _try_spatial_plus_color(build_transpose_gather(ih, iw), "transpose")
            _try_spatial_plus_color(build_anti_transpose_gather(ih, iw), "anti_transpose")
        for angle in [90, 180, 270]:
            _try_spatial_plus_color(build_rotation_gather(angle, ih, iw), f"rot{angle}")
        for direction in ["vertical", "horizontal", "both"]:
            _try_spatial_plus_color(build_flip_gather(direction, ih, iw), f"flip_{direction}")
        # scale / tile / downscale compositions
        for factor in [2, 3, 4]:
            if ih * factor <= H and iw * factor <= W:
                _try_spatial_plus_color(build_scale_gather(factor, ih, iw), f"scale{factor}")
            if ih % factor == 0 and iw % factor == 0:
                oh_x = ih // factor; ow_x = iw // factor
                # downscale gather
                ds_gi = np.full(HW, SAFE_PAD_IDX, dtype=np.int64)
                for r in range(oh_x):
                    for c in range(ow_x):
                        ds_gi[r*W + c] = (r*factor)*W + (c*factor)
                _try_spatial_plus_color(ds_gi, f"downscale{factor}")

    if best_model is not None and elapsed() < time_budget * 0.5:
        # Still try cheap learned conv in case it's cheaper than our composite
        for ks in [1]:
            if elapsed() > time_budget: break
            m = try_learned_conv_fast(train_pairs, all_pairs, kernel_size=ks)
            if m is not None: try_model(m, f"lconv_{ks}")
        return best_model
    if best_model is not None:
        return best_model

    # ---- LEARNED CONV (arc-gen augmented) ----
    for ks in [1, 3]:
        if elapsed() > time_budget: break
        m = try_learned_conv_plus(train_pairs, all_pairs, arc_gen, kernel_size=ks)
        if try_model(m, f"lconv_plus_{ks}"):
            if best_cost < 10000:
                return best_model
    # Also try the original (uses only train_pairs, sometimes converges better)
    for ks in [1, 3]:
        if elapsed() > time_budget: break
        m = try_learned_conv_fast(train_pairs, all_pairs, kernel_size=ks)
        if try_model(m, f"lconv_{ks}"):
            if best_cost < 10000:
                return best_model

    if best_model is not None:
        return best_model

    # Two-layer conv (v8 path)
    for ks1, ks2, hidden in [(3, 1, 16), (3, 1, 24), (3, 3, 16), (5, 1, 12)]:
        if elapsed() > time_budget: break
        m = try_two_layer_conv_fast(train_pairs, all_pairs, ks1=ks1, ks2=ks2, hidden=hidden)
        if try_model(m, f"lconv2_{ks1}_{ks2}_{hidden}"):
            break

    if best_model is not None:
        return best_model

    # ---- V9 DEEPER CNN PROGRESSIVE FALLBACK ----
    if elapsed() < time_budget * 0.95:
        m = try_always_attempt_fallback(train_pairs, all_pairs, arc_gen,
                                        max_aux=min(50, len(arc_gen)))
        if m is not None:
            try_model(m, "deep_fallback")

    if best_model is not None:
        return best_model

    # Constant output
    if info.get("const_output"):
        c_out = grid_to_tensor(train_pairs[0]["output"])
        try_model(make_const_onnx(c_out), "const")

    return best_model

# ============================================================
# I/O
# ============================================================
def save_model(model, path):
    with open(path, "wb") as f:
        f.write(model.SerializeToString())

def find_task_files():
    for d in [TASK_DIR, Path("/kaggle/input"), Path(".")]:
        if d.exists():
            files = list(d.glob("task*.json"))
            if files: return sorted(files)
    task_files = []
    root = "/kaggle/input" if Path("/kaggle/input").exists() else "."
    for r, _, files in os.walk(root):
        for f in files:
            if f.startswith("task") and f.endswith(".json"):
                task_files.append(Path(r) / f)
    return sorted(task_files)

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    task_files = find_task_files()
    if not task_files:
        print("ERROR: no task files"); sys.exit(1)
    print(f"Found {len(task_files)} task files")

    solved = []; failed = []; onnx_files = []
    total_score = 0.0
    start = time.time()

    for task_path in task_files:
        task_name = task_path.stem
        task_num = task_name.replace("task", "")
        try:
            task_data = load_task(task_path)
        except Exception as e:
            print(f"{task_name}: load error {e}")
            failed.append(task_name); continue

        t0 = time.time()
        print(f"Processing {task_name}...", end=" ", flush=True)
        try:
            model = solve_task(task_data, task_name, time_budget=25.0)
        except Exception as e:
            print(f"ERROR: {e}")
            failed.append(task_name); continue
        dt = time.time() - t0

        if model is None:
            print(f"No solution ({dt:.1f}s)")
            failed.append(task_name); continue

        out_path = OUTPUT_DIR / f"task{task_num}.onnx"
        try:
            save_model(model, out_path)
            onnx_files.append(out_path)
            cost = estimate_model_cost(model)
            score = max(1, 25 - math.log(max(1, cost)))
            total_score += score
            print(f"Solved cost={cost} score={score:.1f} ({dt:.1f}s)")
            solved.append(task_name)
        except Exception as e:
            print(f"Save error: {e}")
            failed.append(task_name)

    total = time.time() - start
    print(f"\n{'='*60}")
    print(f"Solved: {len(solved)}/{len(task_files)}  Failed: {len(failed)}")
    print(f"Total local estimated score: {total_score:.1f}  Total time: {total:.1f}s")

    if onnx_files:
        zip_path = OUTPUT_DIR / "submission.zip"
        with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
            for f in onnx_files:
                zf.write(f, f.name)
        print(f"Submission: {zip_path} ({zip_path.stat().st_size/1024:.1f} KB)")